**Navigation** : [carnet source Python](Jauge-03-Action-Wilson-Confinement-Python.ipynb) | [Index de la série](README.md)

# L'action de Wilson sur réseau : la boucle de Wilson et le confinement (C#)

**Série** : Physique / Jauge · **Noyau** : .NET (C#)

Jumeau C# de [Jauge-03 — L'action de Wilson sur réseau : la boucle de Wilson et le confinement (Python)](Jauge-03-Action-Wilson-Confinement-Python.ipynb) : même fil pédagogique, mêmes verdicts — rejoués ici en **.NET natif**, BCL seule, aucun paquet externe ni NuGet.

**Jauge-01** a établi le théorème de Noether : une symétrie continue produit une conservation. **Jauge-02** a construit la dérivée covariante $D_\mu = \partial_\mu + iqA_\mu$, le tenseur $F_{\mu\nu}$, et vérifié l'invariance complète du lagrangien, pour $U(1)$ puis $SU(2)$. Ces constructions sont **perturbatives** : elles parlent de petites oscillations du champ autour du vide.

Ce carnet pose la question que le développement perturbatif ne sait pas trancher : **le confinement** — pourquoi le potentiel entre deux charges statiques croît-il linéairement avec la distance ? La réponse de Kenneth Wilson (1974) passe par une **discrétisation** : sur un réseau, les variables ne sont plus les champs $A_\mu$ mais les **transports parallèles** le long des liens — et une observable unique, la **boucle de Wilson**, sépare la phase confinante de la phase de Coulomb.

Fil du carnet :

1. §1 du continu au réseau : le lien $U_\mu(x) = e^{iagA_\mu(x)}$ comme transport parallèle discret ;
2. §2 la transformation de jauge discrète et le **plaquette**, dont la trace est invariante ;
3. §3 l'**action de Wilson** $S = \beta \sum_p (1 - \frac{1}{N}\mathrm{Re}\,\mathrm{Tr}\, P_p)$ ;
4. §4 la **boucle de Wilson** $W(C)$ : loi d'aire (confinement) contre loi de périmètre (liberté) ;
5. §5 Monte Carlo $U(1)$ en dimension 2, contre la **loi exacte** $\sigma = -\ln(I_1(\beta)/I_0(\beta))$ — hypothèse pré-enregistrée ;
6. §6 la transition de phase en dimension 4 : couplage fort contre phase de Coulomb — contre-témoin ;
7. §7 la boucle de Wilson en Chern–Simons : le polynôme de Jones par la R-matrice, en arithmétique exacte ;
8. §8 l'existence Yang–Mills : l'énoncé formel et son test de non-liberté ;
9. §9 synthèse, §10 quatre exercices.

**L'outil.** Deux arithmétiques, toutes deux dans la bibliothèque standard du .NET :

- le `double` porte tout le calcul flottant : les liens $SU(2)$ et les phases $U(1)$ dans `System.Numerics.Complex`, le Monte Carlo des §5 et §6 à graines fixées, et au §8 les fonctions de Bessel, calculées par quadrature de Gauss–Legendre à 40 nœuds par panneau ;
- le polynôme de Jones du §7 exige l'**exactitude** : la R-matrice de $U_q(\mathfrak{sl}_2)$ n'a que des coefficients en $q$ et $q^{-1}$, donc le carnet manipule des **polynômes de Laurent à coefficients entiers** (`System.Numerics.BigInteger`), multiplie des matrices sur cet anneau et divise **exactement** par $q+q^{-1}$. Aucun arrondi n'entre dans le §7.

**Filière** : SymmetryGroups/Jauge — prolonge [Jauge-02](Jauge-02-Invariance-Jauge-U1-SU2-Python.ipynb).

[Version Python](Jauge-03-Action-Wilson-Confinement-Python.ipynb) : le carnet d'origine, avec numpy, sympy, mpmath, pandas et matplotlib.

## §1. Du continu au réseau : le transport parallèle devient la variable

Sur un réseau de pas $a$, la quantité géométrique naturelle n'est pas la connexion $A_\mu(x)$ elle-même, mais son **intégrale le long d'un lien** :

$$U_\mu(x) = e^{iagA_\mu(x)} \in G, \qquad G = U(1) \text{ ou } SU(2).$$

Trois raisons de préférer les $U_\mu(x)$ aux $A_\mu(x)$ sur le réseau :

- ce sont des **éléments du groupe** — la théorie vit exactement sur $G$, pas sur son algèbre approchée ;
- le transport d'un point à un autre est un **produit de liens**, quelle que soit la finesse du chemin choisi pour le définir ;
- la limite $a \to 0$ se lit comme une propriété de convergence du produit $\prod e^{iagA}$.

La cellule suivante mesure cette convergence dans le cas non abélien : pour une suite de matrices $X_k$ **non commutantes**, le produit $\prod_k e^{iaX_k}$ diffère de $e^{ia\sum_k X_k}$ d'un terme en $O(a^2)$ — l'erreur de commutation, qui s'efface quand le pas devient fin.

In [1]:
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;
using System.Numerics;

// Sorties numeriques au point decimal, comme celles du carnet Python.
CultureInfo.DefaultThreadCurrentCulture = CultureInfo.InvariantCulture;
CultureInfo.CurrentCulture = CultureInfo.InvariantCulture;

// Matrice 2x2 complexe : c'est le type des liens SU(2) et des transformations de jauge locales.
struct M2
{
    public Complex A, B, C, D;
    public M2(Complex a, Complex b, Complex c, Complex d) { A = a; B = b; C = c; D = d; }
    public static M2 Id => new M2(Complex.One, Complex.Zero, Complex.Zero, Complex.One);
    public static M2 Zero => new M2(Complex.Zero, Complex.Zero, Complex.Zero, Complex.Zero);
    public static M2 operator +(M2 x, M2 y) => new M2(x.A + y.A, x.B + y.B, x.C + y.C, x.D + y.D);
    public static M2 operator -(M2 x, M2 y) => new M2(x.A - y.A, x.B - y.B, x.C - y.C, x.D - y.D);
    public static M2 operator *(M2 x, M2 y) => new M2(
        x.A * y.A + x.B * y.C, x.A * y.B + x.B * y.D,
        x.C * y.A + x.D * y.C, x.C * y.B + x.D * y.D);
    public static M2 operator *(M2 x, Complex s) => new M2(x.A * s, x.B * s, x.C * s, x.D * s);
    public static M2 operator *(Complex s, M2 x) => x * s;
    public M2 Dag => new M2(Complex.Conjugate(A), Complex.Conjugate(C),
                            Complex.Conjugate(B), Complex.Conjugate(D));
    public Complex Trace => A + D;
    public double MaxDiff(M2 o) => Math.Max(
        Math.Max(Complex.Abs(A - o.A), Complex.Abs(B - o.B)),
        Math.Max(Complex.Abs(C - o.C), Complex.Abs(D - o.D)));
}

// Deux generateurs non commutants de su(2) : X = i sigma / 2.
var X1 = new M2(Complex.Zero, new Complex(0, 0.5), new Complex(0, 0.5), Complex.Zero);
var X2 = new M2(Complex.Zero, new Complex(0.5, 0), new Complex(-0.5, 0), Complex.Zero);
Console.WriteLine($"commutateur [X1, X2] non nul : {(X1 * X2 - X2 * X1).MaxDiff(M2.Zero) > 1e-3}");

// Exponentielle 2x2 par serie (30 termes).
M2 Expm2(M2 m)
{
    var e = M2.Id;
    var term = M2.Id;
    for (int k = 1; k < 30; k++)
    {
        term = term * m * (1.0 / k);
        e = e + term;
    }
    return e;
}

// Chemin de six pas alternant mu = 1 et mu = 2 : produit des exponentielles contre exponentielle de la somme.
var chemin = new[] { X1, X2, X1, X2, X1, X2 };
var somme = M2.Zero;
foreach (var X in chemin) somme = somme + X;

var erreurs = new List<double>();
foreach (double a in new[] { 0.2, 0.1, 0.05 })
{
    var prod = M2.Id;
    foreach (var X in chemin) prod = prod * Expm2(X * a);
    double err = prod.MaxDiff(Expm2(somme * a));
    erreurs.Add(err);
    Console.WriteLine($"a = {a,5:F3} | prod - exp(somme) : {err,9:E3}");
}
Console.WriteLine($"rapport des erreurs quand a -> a/2 : {erreurs[0] / erreurs[1]:F1} (ordre O(a^2) attendu : ~4.0)");

commutateur [X1, X2] non nul : True


a = 0.200 | prod - exp(somme) : 2.911E-002


a = 0.100 | prod - exp(somme) : 7.444E-003


a = 0.050 | prod - exp(somme) : 1.871E-003


rapport des erreurs quand a -> a/2 : 3.9 (ordre O(a^2) attendu : ~4.0)


### Lecture du résultat

L'écart entre le **produit des exponentielles** et l'**exponentielle de la somme** est le prix de la non-commutativité : il est porté par les commutateurs emboîtés de la formule de Baker–Campbell–Hausdorff, et son ordre est exactement $a^2$ — diviser le pas par deux divise l'erreur par quatre, et le rapport mesuré ci-dessus vaut $3{,}9$, l'asymptotique n'étant atteinte qu'à la limite. L'erreur elle-même décroît d'un facteur 3,9 puis 4,0 à chaque division du pas (2,9·10⁻² → 7,4·10⁻³ → 1,9·10⁻³), ce qui est la signature d'un terme dominant en $a^2$.

Le carnet Python calcule lui aussi cette exponentielle par la même série tronquée à 30 termes, et les flottants du C# sont les mêmes `double` que ceux de numpy : la cellule reproduit les valeurs du carnet Python aux chiffres affichés. Pour $U(1)$, groupe abélien, l'écart est **nul à toute précision** : le produit des phases est la phase de la somme.

## §2. La transformation de jauge discrète et le plaquette

Sur le réseau, une transformation de jauge est une famille $\Omega(x) \in G$ définie **sur les sites**. Elle agit sur les liens en les entourant :

$$U_\mu(x) \;\longmapsto\; U'_\mu(x) = \Omega(x)\, U_\mu(x)\, \Omega^\dagger(x + \hat{\mu}).$$

Le **plaquette** est le plus petit lacet du réseau — un carré élémentaire dans le plan $(\mu, \nu)$ :

$$P_{\mu\nu}(x) = U_\mu(x)\, U_\nu(x+\hat{\mu})\, U^\dagger_\mu(x+\hat{\nu})\, U^\dagger_\nu(x).$$

C'est l'analogue discret de $e^{i a^2 g F_{\mu\nu}(x)}$ : le lacet élémentaire **mesure la courbure**. Sous transformation de jauge, les $\Omega$ intérieurs se télescopent deux à deux et il reste

$$P'_{\mu\nu}(x) = \Omega(x)\, P_{\mu\nu}(x)\, \Omega^\dagger(x),$$

de sorte que la **trace** — l'observable physique — est exactement invariante. La cellule suivante le vérifie dans $SU(2)$, avec des $\Omega$ tirés uniformément (mesure de Haar) via la paramétrisation par quaternions normalisés : quatre gaussiennes normalisées donnent un quaternion unité, donc une matrice $2\times2$ unitaire de déterminant un.

In [2]:
var rng = new Random(42);

// Une gaussienne centree reduite par Box-Muller.
double Gauss(Random r)
{
    double u1 = 1.0 - r.NextDouble();          // (0, 1]
    double u2 = r.NextDouble();
    return Math.Sqrt(-2.0 * Math.Log(u1)) * Math.Cos(2.0 * Math.PI * u2);
}

// Tirage uniforme (Haar) sur SU(2) : quaternion unite -> matrice 2x2.
M2 Su2(Random r)
{
    double a0 = Gauss(r), a1 = Gauss(r), a2 = Gauss(r), a3 = Gauss(r);
    double n = Math.Sqrt(a0 * a0 + a1 * a1 + a2 * a2 + a3 * a3);
    a0 /= n; a1 /= n; a2 /= n; a3 /= n;
    return new M2(new Complex(a0, a3), new Complex(a1, a2),
                  new Complex(-a1, a2), new Complex(a0, -a3));
}

// Quatre liens du plaquette et Omega des quatre sites du carre 0 -> e1 -> e1+e2 -> e2.
var U1 = Su2(rng);
var U2 = Su2(rng);
var O0 = Su2(rng);
var O1 = Su2(rng);
var O2 = Su2(rng);
var O3 = Su2(rng);

var P = U1 * U2 * U1.Dag * U2.Dag;

// Liens transformes Omega(x) U Omega(x + mu)^dag sur les aretes du carre.
var T1 = O0 * U1 * O1.Dag;   // arete 0 -> e1
var T2 = O1 * U2 * O2.Dag;   // arete e1 -> e1 + e2
var T3 = O3 * U1 * O2.Dag;   // arete e2 -> e1 + e2 : U_mu(x + e2)
var T4 = O0 * U2 * O3.Dag;   // arete 0 -> e2 : U_nu(0)
var Pp = T1 * T2 * T3.Dag * T4.Dag;

Console.WriteLine($"Tr P (avant) = {P.Trace.Real}");
Console.WriteLine($"Tr P (apres) = {Pp.Trace.Real}");
Console.WriteLine($"|Tr P - Tr P'| = {Complex.Abs(P.Trace - Pp.Trace):E3}");
Console.WriteLine($"residu P' - Omega(0) P Omega(0)^dag : {Pp.MaxDiff(O0 * P * O0.Dag):E3}");

Tr P (avant) = -0.9421417649201844


Tr P (apres) = -0.9421417649201846


|Tr P - Tr P'| = 2.220E-016


residu P' - Omega(0) P Omega(0)^dag : 2.483E-016


### Lecture du résultat

La trace du plaquette est invariante **à la précision machine** ($2{,}2\times 10^{-16}$), et le plaquette transformé vaut $\Omega(0) P \Omega^\dagger(0)$ à $2{,}5\times 10^{-16}$ près : les transformations des sites intérieurs se sont annulées deux à deux, comme prévu par le télescopage. Toute quantité construite sur des traces de lacets fermés — c'est-à-dire toute observable physique — sera donc insensible au choix de jauge. C'est cette invariance, exacte sur le réseau à tout $a$, que le continu n'obtient qu'à la limite.

Les valeurs numériques diffèrent de celles du carnet Python : le tirage de Haar y vient du générateur PCG64 de numpy, ici de `System.Random`. L'invariance, elle, ne dépend pas du tirage — c'est une identité.

## §3. L'action de Wilson

L'action la plus simple construite sur les plaquettes, invariante de jauge par construction :

$$S_W[U] = \beta \sum_{x} \sum_{\mu < \nu} \left( 1 - \frac{1}{N}\,\mathrm{Re}\,\mathrm{Tr}\, P_{\mu\nu}(x) \right), \qquad \beta = \frac{2N}{g^2}.$$

Pour $U(1)$ ($N=1$), chaque lien porte une phase $\theta_\mu(x) \in \left]-\pi, \pi\right]$ et $P_{\mu\nu} = e^{i\theta_{\mu\nu}}$ avec

$$\theta_{\mu\nu}(x) = \theta_\mu(x) + \theta_\nu(x+\hat{\mu}) - \theta_\mu(x+\hat{\nu}) - \theta_\nu(x),$$

de sorte que $S_W = \beta \sum (1 - \cos \theta_{\mu\nu})$. La cellule suivante vérifie sur un petit réseau $3^4$ que $S_W$ est numériquement inchangé par une transformation de jauge **aléatoire** — l'analogue discret du $\delta \mathcal{L} = 0$ de Jauge-02.

C'est aussi la cellule qui pose l'outillage commun à tout le carnet : un réseau hypercubique périodique en dimension quelconque, avec sa **table de voisins** précalculée — c'est elle qui rendra le Monte Carlo de la dimension 4 supportable.

In [3]:
// Reseau hypercubique periodique en dimension D ; table des voisins precalculee.
sealed class Lattice
{
    public readonly int D;
    public readonly int[] Dim;
    public readonly int N;
    public readonly int[][] Plus;    // Plus[mu][i] = site i + mu
    public readonly int[][] Minus;   // Minus[mu][i] = site i - mu

    public Lattice(params int[] dims)
    {
        D = dims.Length;
        Dim = (int[])dims.Clone();
        N = dims.Aggregate(1, (u, v) => u * v);
        var stride = new int[D];
        stride[0] = 1;
        for (int k = 1; k < D; k++) stride[k] = stride[k - 1] * Dim[k - 1];
        var x = new int[D];
        Plus = Enumerable.Range(0, D).Select(_ => new int[N]).ToArray();
        Minus = Enumerable.Range(0, D).Select(_ => new int[N]).ToArray();
        for (int i = 0; i < N; i++)
        {
            int rest = i;
            for (int k = 0; k < D; k++) { x[k] = rest % Dim[k]; rest /= Dim[k]; }
            for (int mu = 0; mu < D; mu++)
            {
                int old = x[mu];
                x[mu] = (old + 1) % Dim[mu];
                Plus[mu][i] = Enumerable.Range(0, D).Sum(k => x[k] * stride[k]);
                x[mu] = (old - 1 + Dim[mu]) % Dim[mu];
                Minus[mu][i] = Enumerable.Range(0, D).Sum(k => x[k] * stride[k]);
                x[mu] = old;
            }
        }
    }

    public int Site(params int[] x)
    {
        int stride = 1, i = 0;
        for (int k = 0; k < D; k++) { i += x[k] * stride; stride *= Dim[k]; }
        return i;
    }
}

// Angle du plaquette (mu, nu) au site i.
double PlaquetteAngle(double[][] th, Lattice lat, int mu, int nu, int i)
    => th[mu][i] + th[nu][lat.Plus[mu][i]] - th[mu][lat.Plus[nu][i]] - th[nu][i];

// Action de Wilson U(1) : beta * somme sur les plaquettes de (1 - cos theta).
double ActionWil(double[][] th, Lattice lat, double beta)
{
    double s = 0;
    for (int mu = 0; mu < lat.D; mu++)
        for (int nu = mu + 1; nu < lat.D; nu++)
            for (int i = 0; i < lat.N; i++)
                s += 1.0 - Math.Cos(PlaquetteAngle(th, lat, mu, nu, i));
    return beta * s;
}

// Phases de liens tirees uniformement dans ]-pi, pi].
double[][] RandomLinks(Lattice lat, Random r)
{
    var th = new double[lat.D][];
    for (int mu = 0; mu < lat.D; mu++)
    {
        th[mu] = new double[lat.N];
        for (int i = 0; i < lat.N; i++) th[mu][i] = (r.NextDouble() * 2 - 1) * Math.PI;
    }
    return th;
}

// Reseau 3^4, action avant et apres une transformation de jauge aleatoire.
var lat3 = new Lattice(3, 3, 3, 3);
var th = RandomLinks(lat3, rng);
const double beta = 0.6;
double S0 = ActionWil(th, lat3, beta);

var alpha = new double[lat3.N];
for (int i = 0; i < lat3.N; i++) alpha[i] = (rng.NextDouble() * 2 - 1) * Math.PI;
var thT = th.Select(a => (double[])a.Clone()).ToArray();
for (int mu = 0; mu < 4; mu++)
    for (int i = 0; i < lat3.N; i++)
        thT[mu][i] = th[mu][i] + alpha[i] - alpha[lat3.Plus[mu][i]];
double S1 = ActionWil(thT, lat3, beta);

Console.WriteLine($"S_W avant = {S0:F6}");
Console.WriteLine($"S_W apres = {S1:F6}");
Console.WriteLine($"|Delta S_W| = {Math.Abs(S1 - S0)}  (telescopage exact attendu)");

// D'ou vient ce zero ? La transformation deplace chaque lien d'un angle d'ordre un, donc
// presque tous les angles de plaquette changent ; mais chaque terme (1 - cos) ne bouge que
// de quelques 1e-15, tres en dessous de l'ULP de la somme : l'arrondi des additions les absorbe.
// On mesure au lieu de supposer, terme par terme puis sur 400 autres tirages.
int diagPlaquettes = 0, diagAngles = 0, diagTermes = 0;
double diagMaxTheta = 0, diagMaxTerme = 0, diagSommeAbs = 0;
for (int mu = 0; mu < lat3.D; mu++)
    for (int nu = mu + 1; nu < lat3.D; nu++)
        for (int i = 0; i < lat3.N; i++)
        {
            double a0 = PlaquetteAngle(th, lat3, mu, nu, i);
            double a1 = PlaquetteAngle(thT, lat3, mu, nu, i);
            double t0 = 1.0 - Math.Cos(a0), t1 = 1.0 - Math.Cos(a1);
            diagPlaquettes++;
            if (a0 != a1) diagAngles++;
            if (t0 != t1) diagTermes++;
            diagMaxTheta = Math.Max(diagMaxTheta, Math.Abs(a1 - a0));
            diagMaxTerme = Math.Max(diagMaxTerme, Math.Abs(t1 - t0));
            diagSommeAbs += Math.Abs(t1 - t0);
        }
Console.WriteLine();
Console.WriteLine($"diagnostic du zero : ULP de S0 = {Math.BitIncrement(S0) - S0:E3}");
Console.WriteLine($"plaquettes : {diagPlaquettes} | angles changes : {diagAngles} | termes (1 - cos) changes : {diagTermes}");
Console.WriteLine($"max |theta' - theta| = {diagMaxTheta:E3} | max |terme' - terme| = {diagMaxTerme:E3} | somme des |terme' - terme| = {diagSommeAbs:E3}");

// Controle sur 400 tirages, avec un generateur dedie : le bloc ne depend pas des tirages precedents.
var rngCtl = new Random(7);
int ctlNuls = 0, ctlSousUlp = 0;
var ctlEcarts = new List<double>();
for (int essai = 0; essai < 400; essai++)
{
    var thE = RandomLinks(lat3, rngCtl);
    var alE = new double[lat3.N];
    for (int i = 0; i < lat3.N; i++) alE[i] = (rngCtl.NextDouble() * 2 - 1) * Math.PI;
    var thE2 = thE.Select(a => (double[])a.Clone()).ToArray();
    for (int mu = 0; mu < 4; mu++)
        for (int i = 0; i < lat3.N; i++)
            thE2[mu][i] = thE[mu][i] + alE[i] - alE[lat3.Plus[mu][i]];
    double e0 = ActionWil(thE, lat3, beta), e1 = ActionWil(thE2, lat3, beta);
    double d = Math.Abs(e1 - e0);
    if (d == 0) ctlNuls++;
    if (d <= Math.BitIncrement(e0) - e0) ctlSousUlp++;
    ctlEcarts.Add(d);
}
var ctlTries = ctlEcarts.OrderBy(x => x).ToList();
Console.WriteLine();
Console.WriteLine($"controle sur 400 tirages : ecart exactement nul : {ctlNuls} | ecart <= 1 ULP : {ctlSousUlp}");
Console.WriteLine($"ecart median : {ctlTries[200]:E3} | au 9e decile : {ctlTries[360]:E3} | max : {ctlTries[399]:E3}");

S_W avant = 293.091343


S_W apres = 293.091343


|Delta S_W| = 0  (telescopage exact attendu)


diagnostic du zero : ULP de S0 = 5.684E-014


plaquettes : 486 | angles changes : 309 | termes (1 - cos) changes : 259


max |theta' - theta| = 3.553E-015 | max |terme' - terme| = 3.331E-015 | somme des |terme' - terme| = 1.186E-013


controle sur 400 tirages : ecart exactement nul : 210 | ecart <= 1 ULP : 368


ecart median : 0.000E+000 | au 9e decile : 5.684E-014 | max : 1.705E-013


### Lecture du résultat

L'action est rigoureusement invariante : chaque terme $\cos\theta_{\mu\nu}$ voit les phases de jauge de ses quatre sites se compenser en télescopes exacts, exactement comme les $\Omega$ du plaquette $SU(2)$. Le résidu affiché est **exactement nul** sur ce tirage — et ce zéro demande une explication, car la transformation ne déplace pas les liens d'une quantité infime : chaque lien change d'un angle d'ordre un, $\alpha(x)-\alpha(x+\hat\mu)$, et 309 des 486 angles de plaquette changent (259 termes $1-\cos\theta$ sur 486). Mais recalculés en `double`, ces angles ne diffèrent des anciens que de $3{,}6\times 10^{-15}$ au plus, les termes de $3{,}3\times 10^{-15}$ au plus, pour un total de $1{,}2\times 10^{-13}$ sur les 486 plaquettes. Or chaque addition de la somme est arrondie à l'ULP de la somme partielle — $5{,}7\times 10^{-14}$ près de $293$ : une perturbation de quelques $10^{-16}$ par terme est bien plus petite que cet arrondi, et les écarts s'effacent au fil de la somme. Le contrôle sur 400 autres tirages le mesure : écart exactement nul 210 fois, d'au plus un ULP dans 92 % des cas, jamais au-delà de trois ULP ($1{,}7\times 10^{-13}$). Le carnet Python, avec un autre tirage, affiche $5{,}7\times 10^{-14}$ — un ULP, même mécanisme. Ce qui dénoncerait une erreur, c'est un résidu d'ordre $\beta$ : un lien mal transformé changerait l'action d'une quantité d'ordre un.

La théorie de jauge sur réseau est ainsi **complètement définie sans jamais parler de $A_\mu$** : le groupe, ses liens, et une action de traces invariantes. Le paramètre $\beta = 2N/g^2$ joue le rôle d'inverse du couplage — grand $\beta$ = couplage faible = proche du continu.

## §4. La boucle de Wilson : loi d'aire contre loi de périmètre

Soit $C$ un lacet fermé. La **boucle de Wilson** est la moyenne du transport parallèle le long de $C$ :

$$W(C) = \left\langle \frac{1}{N}\, \mathrm{Re}\, \mathrm{Tr} \prod_{\ell \in C} U_\ell \right\rangle.$$

Pour un **rectangle $R \times T$**, deux comportements asymptotiques s'affrontent :

- **loi d'aire** : $W(R, T) \sim e^{-\sigma A}$ avec $A = R \cdot T$ — le coût croît avec la **surface** ; le potentiel entre charges statiques est alors $V(R) = \sigma R$ : **confinement** ;
- **loi de périmètre** : $W \sim e^{-\rho P}$ — le coût croît avec le **périmètre** : pas de force à longue distance, phase de Coulomb (liberté).

La tension de ficelle $\sigma$ est donc **l'observable qui trie les phases**. Le programme de Wilson : mesurer $W$, en tirer $\sigma$, et demander si $\sigma > 0$ (confinement) ou $\sigma = 0$ (liberté). C'est une question **non perturbative** — aucun développement en puissances de $g$ ne peut la trancher — et le réseau la traite par échantillonnage direct de la mesure de Boltzmann $e^{-S_W}$.

## §5. Monte Carlo $U(1)$ en dimension 2 : contre la loi exacte

En dimension **2**, la théorie $U(1)$ compacte est exactement soluble : chaque lien appartient à exactement deux plaquettes, et l'intégration exacte sur les liens donne

$$\langle W(A) \rangle = \left( \frac{I_1(\beta)}{I_0(\beta)} \right)^{A}, \qquad \sigma_{\mathrm{exact}}(\beta) = -\ln \frac{I_1(\beta)}{I_0(\beta)},$$

où $I_0, I_1$ sont les fonctions de Bessel modifiées. Le développement en couplage fort $I_1/I_0 = \beta/2 + O(\beta^3)$ redonne la limite $\sigma \to -\ln(\beta/2)$.

La dimension 2 est donc le **banc d'étalonnage** de tout simulateur : si le Metropolis est correct, la tension mesurée doit coller à $\sigma_{\mathrm{exact}}$. Le simulateur ci-dessous met à jour les liens **en échiquier** : parmi les liens d'une même direction, deux d'entre eux ne partagent un plaquette que si leurs sites de base diffèrent d'un pas — donc de parité opposée. La mise à jour simultanée de tous les liens d'une direction sur une seule parité est ainsi un Metropolis valide ; les agrafes sont recalculées entre les deux passes. L'énergie d'un lien se calcule par ses **agrafes** (staples) — les trois autres liens de chaque plaquette qui le contient.

Les deux fonctions de Bessel $I_0$ et $I_1$ sont calculées ici par leur série entière, sans bibliothèque : c'est la seule dépendance numérique du carnet Python (mpmath) qui disparaît.

In [4]:
// Somme des agrafes de la direction mu : S[i] = somme_nu [ e^{i fwd} + e^{i bwd} ].
// fwd est l'agrafe avant, bwd l'agrafe arriere (le lien y intervient conjugue).
Complex[] StapleSum(double[][] th, Lattice lat, int mu)
{
    var s = new Complex[lat.N];
    for (int nu = 0; nu < lat.D; nu++)
    {
        if (nu == mu) continue;
        for (int i = 0; i < lat.N; i++)
        {
            double fwd = th[nu][lat.Plus[mu][i]] - th[mu][lat.Plus[nu][i]] - th[nu][i];
            double bwd = th[nu][lat.Minus[nu][i]]
                       - th[mu][lat.Minus[nu][i]]
                       - th[nu][lat.Plus[mu][lat.Minus[nu][i]]];
            s[i] += new Complex(Math.Cos(fwd), Math.Sin(fwd));
            s[i] += new Complex(Math.Cos(bwd), Math.Sin(bwd));
        }
    }
    return s;
}

// Angle du rectangle R x T de coin i, directions (0, 1).
double LoopAngle(double[][] th, Lattice lat, int R, int T, int i)
{
    double ang = 0;
    int pa = i;
    for (int a = 0; a < R; a++)
    {
        int pb = pa;
        for (int t = 0; t < T; t++) pb = lat.Plus[1][pb];
        ang += th[0][pa] - th[0][pb];
        pa = lat.Plus[0][pa];
    }
    int pR = i;
    for (int a = 0; a < R; a++) pR = lat.Plus[0][pR];
    int pj = i;
    for (int j = 0; j < T; j++)
    {
        ang += th[1][pR] - th[1][pj];
        pR = lat.Plus[1][pR];
        pj = lat.Plus[1][pj];
    }
    return ang;
}

// Metropolis U(1) en echiquier, dimension quelconque.
(double Acc, Dictionary<(int R, int T), double[]> Hist) SimulateU1(
    Lattice lat, double beta, int nTherm, int nMeas, (int R, int T)[] shapes,
    double delta = 1.0, int seed = 7)
{
    var r = new Random(seed);
    var th = RandomLinks(lat, r);

    // Parite de site : XOR des parites des coordonnees.
    var par = new bool[lat.N];
    var x = new int[lat.D];
    for (int i = 0; i < lat.N; i++)
    {
        int rest = i;
        for (int k = 0; k < lat.D; k++) { x[k] = rest % lat.Dim[k]; rest /= lat.Dim[k]; }
        bool p = false;
        for (int k = 0; k < lat.D; k++) p ^= x[k] % 2 == 1;
        par[i] = p;
    }

    double Sweep()
    {
        int acc = 0, tot = 0;
        for (int mu = 0; mu < lat.D; mu++)
        {
            foreach (bool pari in new[] { true, false })
            {
                // Une seule parite a la fois : deux liens d'une meme direction
                // partagent un plaquette ssi leurs sites different d'un pas, donc
                // sont de parites opposees. Les agrafes sont recalculees entre les passes.
                var S = StapleSum(th, lat, mu);
                for (int i = 0; i < lat.N; i++)
                {
                    if (par[i] != pari) continue;
                    double old = th[mu][i];
                    double prop = old + (r.NextDouble() * 2 - 1) * delta;
                    double dS = -beta * ((Math.Cos(prop) - Math.Cos(old)) * S[i].Real
                                       - (Math.Sin(prop) - Math.Sin(old)) * S[i].Imaginary);
                    if (dS < 0 || Math.Log(Math.Max(r.NextDouble(), 1e-300)) < -dS)
                    {
                        th[mu][i] = prop;
                        acc++;
                    }
                    tot++;
                }
            }
        }
        return tot == 0 ? 0.0 : (double)acc / tot;
    }

    for (int k = 0; k < nTherm; k++) Sweep();
    var hist = shapes.ToDictionary(s => s, s => new double[nMeas]);
    double taux = 0;
    for (int k = 0; k < nMeas; k++)
    {
        taux += Sweep();
        foreach (var s in shapes)
        {
            double somme = 0;
            for (int i = 0; i < lat.N; i++) somme += Math.Cos(LoopAngle(th, lat, s.R, s.T, i));
            hist[s][k] = somme / lat.N;
        }
    }
    return (taux / nMeas, hist);
}

// Ecart type d'echantillon (ddof = 1).
double Std(double[] v)
{
    double m = v.Average();
    return Math.Sqrt(v.Sum(z => (z - m) * (z - m)) / (v.Length - 1));
}

// Erreur statistique par blocs : l'autocorrelation du Monte Carlo est de quelques balayages.
double ErrBlocs(double[] v, int taille)
{
    var blocs = Enumerable.Range(0, v.Length / taille)
                          .Select(k => v.Skip(k * taille).Take(taille).Average())
                          .ToArray();
    return Std(blocs) / Math.Sqrt(blocs.Length);
}

### Étape 0 — valider les agrafes contre l'action brute

Avant de mesurer quoi que ce soit, l'ingrédient critique — la formule d'agrafe qui donne l'énergie locale d'un lien — est confronté à la définition **brute** de l'action : sur une configuration aléatoire, changer un lien et vérifier que le $\Delta S$ prédit par l'agrafe égalise $S_{\mathrm{après}} - S_{\mathrm{avant}}$ calculé en recomptant toute l'action. Si l'agrafe arrière (celle où le lien intervient conjugué) était fausse, l'écart serait ici d'ordre $\beta$ — impossible à rater.

In [5]:
var lat6 = new Lattice(6, 6);
var r5 = new Random(3);
var thTest = RandomLinks(lat6, r5);
const double betaT = 0.9;

const int muT = 1;
int site = lat6.Site(2, 3);
double sAvant = ActionWil(thTest, lat6, betaT);
var stap = StapleSum(thTest, lat6, muT)[site];

// Modification proposee du lien theta_mu(site).
double oldAng = thTest[muT][site];
double newAng = oldAng + 1.1;
double dSAgr = -betaT * ((Math.Cos(newAng) - Math.Cos(oldAng)) * stap.Real
                       - (Math.Sin(newAng) - Math.Sin(oldAng)) * stap.Imaginary);
thTest[muT][site] = newAng;
double sApres = ActionWil(thTest, lat6, betaT);
thTest[muT][site] = oldAng;   // restauration

Console.WriteLine($"delta S predit par les agrafes  : {dSAgr:F12}");
Console.WriteLine($"delta S recompte (action brute) : {sApres - sAvant:F12}");
Console.WriteLine($"ecart : {Math.Abs(dSAgr - (sApres - sAvant)):E2}");

delta S predit par les agrafes  : -1.434514464961


delta S recompte (action brute) : -1.434514464961


ecart : 2.89E-015


### Lecture du résultat

Le $\Delta S$ de l'agrafe et celui de l'action recomptée coïncident à la précision machine : la décomposition en agrafes — y compris l'agrafe arrière, où le lien intervient conjugué — reproduit exactement l'action de Wilson. Le simulateur peut maintenant être confronté à la loi exacte.

C'est ce contrôle qui rend le reste crédible : une erreur de signe dans l'agrafe arrière, ou un oubli de la permutation des sites, se verrait ici à la première décimale, alors qu'il serait invisible dans une moyenne de boucles.

### Hypothèse pré-enregistrée

Avant de lancer la mesure, l'hypothèse est **fixée par écrit** : à $\beta = 0{,}6$, sur un réseau $12 \times 12$ (300 balayages de thermalisation, 6000 mesures), la tension déduite de la plus petite boucle,

$$\sigma_{\mathrm{mes}} = -\ln \langle W(1,1) \rangle,$$

doit vérifier $\left|\sigma_{\mathrm{mes}} - \sigma_{\mathrm{exact}}\right| / \sigma_{\mathrm{exact}} < 0{,}05$, avec $\sigma_{\mathrm{exact}} = -\ln(I_1(0{,}6)/I_0(0{,}6))$ calculée par la série entière des fonctions de Bessel.

Une erreur d'implémentation (agrafes fausses, échiquier invalide, signe de $\Delta S$, mesure de boucle fausse) déplace la mesure bien au-delà de cette tolérance — l'hypothèse sera **rejetée** si c'est le cas. Le témoin du couplage fort, $-\ln(\beta/2) = 1{,}2040$, est rappelé pour situer l'écart que la loi exacte doit discriminer.

In [6]:
// Fonctions de Bessel modifiees I0 et I1 par leur serie entiere.
double Factorial(int n)
{
    double f = 1;
    for (int k = 2; k <= n; k++) f *= k;
    return f;
}

double BesselI(int n, double x)
{
    double term = Math.Pow(x / 2, n) / Factorial(n);
    double sum = term;
    for (int k = 1; k < 300; k++)
    {
        term *= (x / 2) * (x / 2) / (k * (k + n));
        sum += term;
        if (Math.Abs(term) < 1e-18 * Math.Abs(sum)) break;
    }
    return sum;
}

double SigmaExact2D(double b) => -Math.Log(BesselI(1, b) / BesselI(0, b));

Console.WriteLine("HYPOTHESE PRE-ENREGISTREE (avant toute mesure) :");
Console.WriteLine($"  cible    : sigma_exact(0.6) = -ln(I1/I0)   = {SigmaExact2D(0.6):F4}");
Console.WriteLine($"  temoin   : couplage fort -ln(beta/2)       = {-Math.Log(0.3):F4}");
Console.WriteLine("  seuil    : |sigma_mes - sigma_exact| / sigma_exact < 0.05");
Console.WriteLine("  protocole: reseau 12x12, beta=0.6, 300 therm + 6000 mesures, sigma = -ln <W(1,1)>");

HYPOTHESE PRE-ENREGISTREE (avant toute mesure) :


  cible    : sigma_exact(0.6) = -ln(I1/I0)   = 1.2474


  temoin   : couplage fort -ln(beta/2)       = 1.2040


  seuil    : |sigma_mes - sigma_exact| / sigma_exact < 0.05


  protocole: reseau 12x12, beta=0.6, 300 therm + 6000 mesures, sigma = -ln <W(1,1)>


In [7]:
var sw = System.Diagnostics.Stopwatch.StartNew();
var (acc06, h06) = SimulateU1(new Lattice(12, 12), 0.6, 300, 6000,
                             new[] { (R: 1, T: 1) }, seed: 7);
var w11 = h06[(1, 1)];

double wMean = w11.Average();
double wErr = ErrBlocs(w11, 150);
double sigmaMes = -Math.Log(wMean);
double cible = SigmaExact2D(0.6);
double rel = Math.Abs(sigmaMes - cible) / cible;

Console.WriteLine($"taux d'acceptation moyen : {acc06:F2}");
Console.WriteLine($"<W(1,1)> = {wMean:F5} +/- {wErr:F5}   (6000 mesures, {w11.Length / 150} blocs)");
Console.WriteLine($"sigma_mes   = -ln <W(1,1)>            = {sigmaMes:F4}");
Console.WriteLine($"sigma_exact = -ln(I1(0.6)/I0(0.6))    = {cible:F4}");
Console.WriteLine($"ecart relatif = {rel * 100:F2} %  -> seuil 5 % : {(rel < 0.05 ? "CONFIRMEE" : "REJECTEE")}");
Console.WriteLine($"duree : {sw.Elapsed.TotalSeconds:F1} s");

taux d'acceptation moyen : 0.88


<W(1,1)> = 0.28610 +/- 0.00135   (6000 mesures, 40 blocs)


sigma_mes   = -ln <W(1,1)>            = 1.2514


sigma_exact = -ln(I1(0.6)/I0(0.6))    = 1.2474


ecart relatif = 0.33 %  -> seuil 5 % : CONFIRMEE


duree : 0.4 s


### Lecture du résultat

La tension mesurée s'aligne sur la loi exacte de Bessel à l'intérieur du seuil pré-enregistré : le simulateur (agrafes, échiquier, Metropolis) est validé par une **prédiction fermée**, pas par simple plausibilité. L'écart entre la cible et le témoin $-\ln(\beta/2)$ est la correction non triviale en $O(\beta^2)$ que le développement en couplage fort omet : à $\beta = 0{,}6$ elle atteint déjà 3,5 %, et le seuil de 5 % est choisi pour laisser passer le bruit statistique sans laisser passer une erreur de simulateur.

Une note sur les nombres : le tirage initial et les propositions viennent de `System.Random`, pas du PCG64 de numpy ; les valeurs mesurées diffèrent donc un peu du carnet Python, qui trouve $\langle W(1{,}1)\rangle = 0{,}28654$ et $\sigma = 1{,}2499$ : l'écart entre les deux tensions, $1{,}5$ millième, reste sous le tiers de la barre d'erreur. Ce qui est comparé — la loi exacte et le seuil pré-enregistré — ne dépend pas du générateur.

La loi d'**aire**, elle, se lit sur des boucles plus grandes — c'est l'objet de la cellule suivante.

### La loi d'aire à $\beta = 1$

À $\beta = 0{,}6$, la tension est forte et les boucles d'aire $\geq 4$ tombent sous le bruit : mesurer une loi exponentielle, c'est choisir son $\beta$ pour que le signal survive. À $\beta = 1$, $\sigma$ est plus petit, et les rectangles $1\times1$, $1\times2$, $1\times3$ restent mesurables dans le même temps de calcul.

La pente de $\ln\langle W\rangle$ contre l'aire est ajustée par moindres carrés sur les trois points, et comparée à $-\sigma_{\mathrm{exact}}(1{,}0)$ — aucune constante ajustable.

In [8]:
var (acc10, h10) = SimulateU1(new Lattice(12, 12), 1.0, 300, 6000,
                              new[] { (R: 1, T: 1), (R: 1, T: 2), (R: 1, T: 3) }, seed: 17);
double cible10 = SigmaExact2D(1.0);

var formes = new[] { (R: 1, T: 1), (R: 1, T: 2), (R: 1, T: 3) };
var aires = new double[3];
var lnW = new double[3];
var lnWErr = new double[3];
for (int k = 0; k < 3; k++)
{
    var v = h10[formes[k]];
    double m = v.Average();
    double e = ErrBlocs(v, 150);
    aires[k] = formes[k].R * formes[k].T;
    lnW[k] = Math.Log(m);
    lnWErr[k] = e / m;
    Console.WriteLine($"W({formes[k].R},{formes[k].T}) : <W> = {m:F4} +/- {e:F4} | A = {aires[k]} | ln W = {lnW[k]:F3} +/- {e / m:F3}");
}

// Moindres carres sur trois points : pente de ln W = f(A).
double pente = Enumerable.Range(0, 3).Sum(k => (aires[k] - aires.Average()) * (lnW[k] - lnW.Average()))
             / Enumerable.Range(0, 3).Sum(k => (aires[k] - aires.Average()) * (aires[k] - aires.Average()));
Console.WriteLine();
Console.WriteLine($"pente ajustee ln W = f(A)        : {pente:F3}");
Console.WriteLine($"prediction exacte -sigma(1.0)    : {-cible10:F3}");

W(1,1) : <W> = 0.4469 +/- 0.0014 | A = 1 | ln W = -0.805 +/- 0.003


W(1,2) : <W> = 0.1999 +/- 0.0014 | A = 2 | ln W = -1.610 +/- 0.007


W(1,3) : <W> = 0.0893 +/- 0.0018 | A = 3 | ln W = -2.416 +/- 0.020


pente ajustee ln W = f(A)        : -0.805


prediction exacte -sigma(1.0)    : -0.807


### Lecture du résultat

Les trois points s'alignent sur la droite $-\sigma_{\mathrm{exact}} A$ : le coût d'une boucle croît avec son **aire**, pas avec son périmètre. La pente ajustée et la prédiction exacte coïncident à mieux que le pour cent, sans aucun paramètre libre.

Le graphique du carnet Python est ici rendu en texte — la bibliothèque standard du .NET n'a pas de tracé — mais la lecture est la même : la colonne des mesures et celle de la droite exacte doivent se superposer.

In [9]:
// Graphique textuel : 'o' = mesure, '-' = droite exacte -sigma * A,
// '*' = les deux confondues a moins d'un demi-pas vertical.
int lignes = 9;
double bas = Math.Min(lnW.Min(), -cible10 * 3) - 0.15;
double haut = 0.0;
var grille = new char[lignes, 3];
for (int i = 0; i < lignes; i++)
    for (int j = 0; j < 3; j++) grille[i, j] = ' ';

double pas = (haut - bas) / (lignes - 1);
for (int i = 0; i < lignes; i++)
{
    double valeur = haut - i * pas;
    for (int k = 0; k < 3; k++)
    {
        bool mesure = Math.Abs(valeur - lnW[k]) <= pas / 2;
        bool exacte = Math.Abs(valeur + cible10 * aires[k]) <= pas / 2;
        grille[i, k] = mesure && exacte ? '*' : mesure ? 'o' : exacte ? '-' : ' ';
    }
}

Console.WriteLine("ln <W>        o = mesure   - = droite exacte   * = confondues");
for (int i = 0; i < lignes; i++)
{
    double valeur = haut - i * pas;
    Console.WriteLine($"{valeur,7:F2} | {grille[i, 0]}   {grille[i, 1]}   {grille[i, 2]}");
}
Console.WriteLine("        +---+---+---");
Console.WriteLine("          A=1 A=2 A=3");

ln <W>        o = mesure   - = droite exacte   * = confondues


   0.00 |          


  -0.32 |          


  -0.64 |          


  -0.96 | *        


  -1.28 |          


... (2 outputs left out)

  -2.25 |          


  -2.57 |         *


        +---+---+---


          A=1 A=2 A=3


## §6. La transition de phase en dimension 4 : le contre-témoin

La dimension 2 confine **à tout $\beta$** — il n'y a pas de phase concurrente. La question du confinement n'est tranchante qu'en dimension **4**, où la théorie $U(1)$ compacte possède une véritable **transition de phase** vers $\beta_c \approx 1{,}01$ :

- $\beta < \beta_c$ (couplage fort) : phase confinante, $\chi(1,1) = -\ln\langle W(1,1)\rangle$ d'ordre unité ;
- $\beta > \beta_c$ : **phase de Coulomb** — le photon devient libre, les charges s'écrantent, $\chi(1,1)$ s'effondre vers 0.

La cellule compare $\beta = 0{,}6$ (couplage fort) à $\beta = 1{,}2$, bien au-delà de la transition.

C'est le contre-témoin du carnet : la même observable, mesurée par le même simulateur, doit **basculer** entre les deux phases. Si le simulateur ne mesurait qu'un artefact, la bascule n'aurait aucune raison d'apparaître entre deux phases que la théorie — établie par ailleurs — place de part et d'autre de $\beta_c$. Deux points de couplage ne localisent pas la transition, ils la traversent.

### Protocole pré-enregistré

Les trois critères sont fixés ici, **avant** la cellule de simulation qui produira les mesures :

| Critère | Signature attendue |
|---|---|
| $\chi(1,1)$ à $\beta = 0{,}6$ | $> 0{,}8$ : ordre unitaire en phase confinante |
| $\chi(1,1)$ à $\beta = 1{,}2$ | $< 0{,}35$ : effondrement en phase de Coulomb |
| Rapport $\chi(0{,}6)/\chi(1{,}2)$ | $> 3$ : bascule franche entre les deux phases |

La cellule suivante exécute le Monte Carlo sans modifier ces seuils ; celle d'après évalue leur conjonction sur les mesures obtenues.

In [10]:
var sw4 = System.Diagnostics.Stopwatch.StartNew();
var chi4 = new Dictionary<double, double>();
foreach (var (beta4, seed4) in new[] { (0.6, 11), (1.2, 12) })
{
    var (a4, h4) = SimulateU1(new Lattice(6, 6, 6, 6), beta4, 200, 1000,
                              new[] { (R: 1, T: 1) }, seed: seed4);
    double w = h4[(1, 1)].Average();
    chi4[beta4] = -Math.Log(w);
    Console.WriteLine($"beta = {beta4:F2} | <W(1,1)> = {w:F4} | chi(1,1) = {chi4[beta4]:F4} | accept. {a4:F2}");
}

double rapport = chi4[0.6] / chi4[1.2];
Console.WriteLine($"rapport chi(fort)/chi(Coulomb) = {rapport:F2}");
Console.WriteLine($"duree : {sw4.Elapsed.TotalSeconds:F1} s");

beta = 0.60 | <W(1,1)> = 0.2942 | chi(1,1) = 1.2236 | accept. 0.80


beta = 1.20 | <W(1,1)> = 0.7522 | chi(1,1) = 0.2848 | accept. 0.58


rapport chi(fort)/chi(Coulomb) = 4.30


duree : 5.6 s


In [11]:
// Evaluation de la conjonction des trois criteres pre-enregistres.
var criteres = new (string Nom, double Valeur, bool Passe)[]
{
    ("chi(1,1) a beta=0.6 > 0.8", chi4[0.6], chi4[0.6] > 0.8),
    ("chi(1,1) a beta=1.2 < 0.35", chi4[1.2], chi4[1.2] < 0.35),
    ("rapport chi(0.6)/chi(1.2) > 3", rapport, rapport > 3),
};

Console.WriteLine($"{"Controle",-32} {"Mesure",8}  {"Passe",5}");
Console.WriteLine(new string('-', 50));
foreach (var c in criteres)
    Console.WriteLine($"{c.Nom,-32} {c.Valeur,8:F3}  {c.Passe,5}");
Console.WriteLine(new string('-', 50));
Console.WriteLine("Bascule de phase detectee : " + (criteres.All(c => c.Passe) ? "OUI" : "NON"));

Controle                           Mesure  Passe


--------------------------------------------------


chi(1,1) a beta=0.6 > 0.8           1.224   True


chi(1,1) a beta=1.2 < 0.35          0.285   True


rapport chi(0.6)/chi(1.2) > 3       4.297   True


--------------------------------------------------


Bascule de phase detectee : OUI


### Lecture du résultat

La même observable, le même code, deux valeurs de $\beta$ : $-\ln\langle W(1,1)\rangle$ chute d'un **facteur plusieurs** entre couplage fort et phase de Coulomb. C'est la signature demandée — la boucle de Wilson **trie les phases**.

Cette chute est **cohérente avec une transition connue par ailleurs à $\beta_c \approx 1{,}01$** pour le $U(1)$ compact en dimension 4, **que ce carnet ne démontre pas** : deux points de couplage ne localisent pas une transition. En QCD ($SU(3)$, dimension 4), la boucle de Wilson, combinée en rapports de Creutz, est l'outil qui a établi numériquement le confinement des quarks : le réseau ne fait pas qu'illustrer la théorie de jauge, il répond à une question que le développement perturbatif ne peut pas poser.

## §7. La même boucle de Wilson, en théorie topologique : le polynôme de Jones

Changeons de théorie sans changer d'observable. En **Chern–Simons** $SU(2)$ au niveau $k$, l'action ne dépend d'aucune métrique :

$$S_{\mathrm{CS}}[A] = \frac{k}{4\pi}\int \mathrm{Tr}\left(A \wedge dA + \frac{2}{3} A \wedge A \wedge A\right).$$

La boucle de Wilson y est le même objet qu'au §4 — $\langle W(C)\rangle = \left\langle \mathrm{Tr}\, P \exp \oint_C A \right\rangle$ — mais le résultat n'est plus une fonction de l'aire : c'est un **invariant topologique**. Witten (1989) a montré qu'il coïncide avec le **polynôme de Jones** du nœud $C$, évalué en

$$t = e^{2\pi i/(k+2)}.$$

Le décalage $k+2$ est la charge quantique de la théorie ; $q = e^{2\pi i/(k+2)}$ est le paramètre de déformation quantique, et la symétrie algébrique sous-jacente est le groupe quantique $U_q(\mathfrak{sl}_2)$.

**Le calcul, sans équations fonctionnelles.** Trois faits algébriques suffisent, et chacun est vérifiable exactement :

1. **Alexander** — tout nœud est la fermeture d'une **tresse** $\hat\beta$ sur $n$ brins.
2. **La R-matrice** $R$ de $U_q(\mathfrak{sl}_2)$ dans la représentation de spin $\frac12$ est une matrice $4\times4$ à coefficients en $q^{\pm1}$ ; elle satisfait l'équation de **Yang–Baxter** $R_{12}R_{23}R_{12} = R_{23}R_{12}R_{23}$ et la relation de **Hecke** $R - R^{-1} = (q - q^{-1})I$. C'est elle qui fait agir un croisement : un brin croisé se lit $R$, l'autre $R^{-1}$.
3. **La trace quantique** — la fermeture contracte l'espace des états $(\mathbb{C}^2)^{\otimes n}$ par $\mathrm{Tr}_q(\rho) = \sum_i \rho_{ii}\, q^{\#(+1) - \#(-1)}$, la torsion par le Cartan ; puis on divise par $[2]_q = q+q^{-1}$ et on corrige l'enlacement par $q^{-2w}$. Ce sont les deux **stabilités de Markov**.

Aucun de ces trois ingrédients n'a besoin d'analyse : tout se calcule dans l'anneau $\mathbb{Z}[q, q^{-1}]$ — les polynômes de Laurent à coefficients entiers. C'est ce que fait la cellule suivante, avec `BigInteger` : **aucun arrondi n'entre dans cette section**.

In [12]:
// Polynome de Laurent a coefficients entiers : l'anneau Z[q, q^-1], exact.
// Remplace sympy : les exposants sont des entiers relatifs, les coefficients des BigInteger.
sealed class LP
{
    public readonly SortedDictionary<int, BigInteger> C = new SortedDictionary<int, BigInteger>();

    public LP() { }
    public bool IsZero => C.Count == 0;
    public int MinE => C.Count == 0 ? 0 : C.Keys.First();
    public int MaxE => C.Count == 0 ? 0 : C.Keys.Last();
    public BigInteger this[int e] => C.TryGetValue(e, out var v) ? v : BigInteger.Zero;

    public static LP Monom(int e, BigInteger c)
    {
        var p = new LP();
        if (!c.IsZero) p.C[e] = c;
        return p;
    }
    public static LP Q(int e = 1) => Monom(e, BigInteger.One);
    public static LP One => Monom(0, BigInteger.One);

    public static LP operator +(LP a, LP b)
    {
        var r = new LP();
        foreach (var kv in a.C) r.C[kv.Key] = kv.Value;
        foreach (var kv in b.C)
        {
            var v = r[kv.Key] + kv.Value;
            if (v.IsZero) r.C.Remove(kv.Key); else r.C[kv.Key] = v;
        }
        return r;
    }
    public static LP operator -(LP a, LP b) => a + (-b);
    public static LP operator -(LP a)
    {
        var r = new LP();
        foreach (var kv in a.C) r.C[kv.Key] = -kv.Value;
        return r;
    }
    public static LP operator *(LP a, LP b)
    {
        var r = new LP();
        foreach (var x in a.C)
            foreach (var y in b.C)
            {
                int e = x.Key + y.Key;
                var v = r[e] + x.Value * y.Value;
                if (v.IsZero) r.C.Remove(e); else r.C[e] = v;
            }
        return r;
    }
    public static LP operator *(LP a, BigInteger s)
    {
        if (s.IsZero) return new LP();
        var r = new LP();
        foreach (var kv in a.C) r.C[kv.Key] = kv.Value * s;
        return r;
    }

    public LP Shift(int k)
    {
        var r = new LP();
        foreach (var kv in C) r.C[kv.Key + k] = kv.Value;
        return r;
    }
    public LP Pow(int n)
    {
        var r = One;
        for (int i = 0; i < n; i++) r = r * this;
        return r;
    }
    // Substitution t -> t^-1 : renverse tous les exposants.
    public LP Reverse()
    {
        var r = new LP();
        foreach (var kv in C) r.C[-kv.Key] = kv.Value;
        return r;
    }
    public bool Same(LP o) => C.Count == o.C.Count
        && C.All(kv => o.C.TryGetValue(kv.Key, out var v) && v == kv.Value);
    public Complex Eval(Complex z)
    {
        var s = Complex.Zero;
        foreach (var kv in C) s += new Complex((double)kv.Value, 0) * Complex.Pow(z, kv.Key);
        return s;
    }

    // Division exacte par un diviseur unitaire (coefficient de plus haut degre +/-1).
    // Algorithme : on decale numerateur ET diviseur pour amener l'exposant MAXIMAL du
    // diviseur a zero ; le diviseur devient unitaire en tete, et chaque etape elimine
    // l'exposant maximal courant du numerateur -- qui decroit strictement. Un quotient
    // de Laurent peut avoir des exposants NEGATIFS : normaliser par l'exposant minimal
    // ne suffirait pas (le cas q + q^-1 le montre). Si le reste n'est pas nul, la
    // division n'est pas exacte : on leve.
    public static LP DivExact(LP a, LP b)
    {
        if (b.IsZero) throw new DivideByZeroException("diviseur nul");
        int decal = b.MaxE;
        var bb = b.Shift(-decal);
        var lead = bb[bb.MaxE];
        if (BigInteger.Abs(lead) != BigInteger.One) throw new ArithmeticException("diviseur non unitaire");
        var num = a.Shift(-decal);
        var quot = new LP();
        int garde = 0;
        while (!num.IsZero)
        {
            if (++garde > 10000) throw new ArithmeticException("division non exacte (boucle)");
            int e = num.MaxE - bb.MaxE;
            var c = num[num.MaxE] / lead;
            quot = quot + Monom(e, c);
            num = num - bb.Shift(e) * c;
        }
        return quot;
    }

    public override string ToString() => ToString("q");

    /// <summary>Affichage dans une autre variable : le meme anneau sert en q et en t.</summary>
    public string ToString(string variable)
    {
        if (IsZero) return "0";
        var parts = new List<string>();
        foreach (var kv in C.OrderByDescending(z => z.Key))
        {
            var c = kv.Value;
            string mag = BigInteger.Abs(c).IsOne && kv.Key != 0 ? "" : BigInteger.Abs(c).ToString();
            string var = kv.Key == 0 ? "" : kv.Key == 1 ? variable : variable + "^" + kv.Key;
            string term = mag + var;
            if (parts.Count == 0) parts.Add((c.Sign < 0 ? "-" : "") + term);
            else parts.Add((c.Sign < 0 ? " - " : " + ") + term);
        }
        return string.Concat(parts);
    }
}

Console.WriteLine("anneau Z[q, q^-1] pret : " + (LP.Q(2) + LP.Q(-2)).ToString());

anneau Z[q, q^-1] pret : q^2 + q^-2


### L'anneau exact

Le type `LP` est un dictionnaire exposant → coefficient. Trois opérations suffisent à toute la section : somme, produit (convolution des exposants) et **division exacte**. La division longue élimine l'exposant **maximal** à chaque étape, ce qui garantit la terminaison ; si un reste non nul subsiste, elle lève au lieu de rendre un quotient approché — c'est ce qui rend la vérification honnête : une erreur de signe ou d'exposant se manifesterait par une exception, pas par un polynôme plausible.

La sortie ci-dessus, `q^-2 + q^2`, est déjà écrite dans la convention d'affichage du carnet (exposants décroissants, signes explicites).

Le même choix que le carnet Python, avec un autre outil : là-bas `sympy` portait l'arithmétique symbolique ; ici `System.Numerics.BigInteger` et un dictionnaire trié la portent, sans aucune dépendance externe. Le résultat — l'exactitude — est identique, et c'est elle qui compte : un polynôme de Jones calculé en flottant ne prouverait rien.

In [13]:
// Matrice d'elements de Z[q, q^-1].
sealed class LPM
{
    public readonly int Nr, Nc;
    public readonly LP[,] E;

    public LPM(int r, int c)
    {
        Nr = r; Nc = c;
        E = new LP[r, c];
        for (int i = 0; i < r; i++) for (int j = 0; j < c; j++) E[i, j] = new LP();
    }
    public static LPM Eye(int n)
    {
        var m = new LPM(n, n);
        for (int i = 0; i < n; i++) m.E[i, i] = LP.One;
        return m;
    }
    public static LPM operator *(LPM a, LPM b)
    {
        if (a.Nc != b.Nr) throw new ArgumentException("dimensions incompatibles");
        var r = new LPM(a.Nr, b.Nc);
        for (int i = 0; i < a.Nr; i++)
            for (int j = 0; j < b.Nc; j++)
            {
                var s = new LP();
                for (int k = 0; k < a.Nc; k++) s = s + a.E[i, k] * b.E[k, j];
                r.E[i, j] = s;
            }
        return r;
    }
    public static LPM operator -(LPM a, LPM b)
    {
        var r = new LPM(a.Nr, a.Nc);
        for (int i = 0; i < a.Nr; i++) for (int j = 0; j < a.Nc; j++) r.E[i, j] = a.E[i, j] - b.E[i, j];
        return r;
    }
    // Produit de Kronecker : (A x B)[i*rb + k, j*cb + l] = A[i,j] B[k,l].
    public static LPM Kron(LPM a, LPM b)
    {
        var r = new LPM(a.Nr * b.Nr, a.Nc * b.Nc);
        for (int i = 0; i < a.Nr; i++)
            for (int j = 0; j < a.Nc; j++)
                for (int k = 0; k < b.Nr; k++)
                    for (int l = 0; l < b.Nc; l++)
                        r.E[i * b.Nr + k, j * b.Nc + l] = a.E[i, j] * b.E[k, l];
        return r;
    }
    public LP Trace
    {
        get
        {
            var s = new LP();
            for (int i = 0; i < Nr; i++) s = s + E[i, i];
            return s;
        }
    }
    public bool IsZero
    {
        get
        {
            for (int i = 0; i < Nr; i++) for (int j = 0; j < Nc; j++) if (!E[i, j].IsZero) return false;
            return true;
        }
    }
    // Determinant par developpement en cofacteurs (4x4 : 24 termes).
    public static LP Det(LPM m)
    {
        if (m.Nr == 1) return m.E[0, 0];
        var s = new LP();
        for (int j = 0; j < m.Nc; j++)
        {
            if (m.E[0, j].IsZero) continue;
            var minor = new LPM(m.Nr - 1, m.Nc - 1);
            for (int i = 1; i < m.Nr; i++)
            {
                int kk = 0;
                for (int k = 0; k < m.Nc; k++)
                    if (k != j) { minor.E[i - 1, kk] = m.E[i, k]; kk++; }
            }
            s = s + (j % 2 == 0 ? m.E[0, j] * Det(minor) : -(m.E[0, j] * Det(minor)));
        }
        return s;
    }
}

// R-matrice de U_q(sl_2), representation de spin 1/2.
// R = [[q,0,0,0],[0,0,1,0],[0,1,q-q^-1,0],[0,0,0,q]].
var qq = LP.Q(1);
var qm1 = LP.Q(-1);
var delta = qq - qm1;

var Rmat = new LPM(4, 4);
Rmat.E[0, 0] = qq;
Rmat.E[1, 2] = LP.One;
Rmat.E[2, 1] = LP.One;
Rmat.E[2, 2] = delta;
Rmat.E[3, 3] = qq;

// R^-1 = R - (q - q^-1) I  (relation de Hecke).
var Rinv = new LPM(4, 4);
var hecke = new LPM(4, 4);
for (int i = 0; i < 4; i++)
{
    hecke.E[i, i] = delta;
    for (int j = 0; j < 4; j++)
        Rinv.E[i, j] = Rmat.E[i, j] - (i == j ? delta : new LP());
}

Console.WriteLine($"R . R^-1 = I                : {(Rmat * Rinv - LPM.Eye(4)).IsZero}");
Console.WriteLine($"R - R^-1 = (q - q^-1) I     : {(Rmat - Rinv - hecke).IsZero}");
Console.WriteLine($"trace R = {Rmat.Trace}      (attendu : 3q - q^-1)");
Console.WriteLine($"det R   = {LPM.Det(Rmat)}   (attendu : -q^2)");

R . R^-1 = I                : True


R - R^-1 = (q - q^-1) I     : True


trace R = 3q - q^-1      (attendu : 3q - q^-1)


det R   = -q^2   (attendu : -q^2)


### Vérification de l'algèbre

Deux identités sont contrôlées ici, exactement, sur des polynômes de Laurent :

- $R\cdot R^{-1} = I$ — l'inverse est bien $R - (q-q^{-1})I$ ;
- la **relation de Hecke** $R - R^{-1} = (q-q^{-1})I$ — c'est elle qui donne au croisement son polynôme caractéristique à deux valeurs propres ;
- le spectre, lu sur la trace et le déterminant : $\mathrm{Tr}\,R = 3q - q^{-1}$ et $\det R = -q^{2}$ — soit les valeurs propres $q$ (multiplicité 3) et $-q^{-1}$ (multiplicité 1), exactement.

C'est le squelette algébrique du croisement : deux projections d'un même nœud ne diffèrent jamais que par ces substitutions. L'équation de Yang–Baxter, testée dans la cellule qui suit, est la compatibilité de deux croisements sur trois brins — sans elle, l'action des tresses ne serait pas un homomorphisme et le « polynôme » dépendrait du diagramme.

In [14]:
// Equation de Yang-Baxter (tresse) : R12 R23 R12 = R23 R12 R23 sur trois brins.
var I2 = LPM.Eye(2);
var R12 = LPM.Kron(LPM.Kron(Rmat, I2), I2);
var R23 = LPM.Kron(LPM.Kron(I2, Rmat), I2);
Console.WriteLine($"Yang-Baxter R12 R23 R12 = R23 R12 R23 : {(R12 * R23 * R12 - R23 * R12 * R23).IsZero}");

// Operateur de tresse : produit des R (ou R^-1) sur les brins (i, i+1).
LPM BraidOperator(int n, int[] word)
{
    var output = LPM.Eye(1 << n);
    foreach (int s in word)
    {
        int i = Math.Abs(s) - 1;
        var local = s > 0 ? Rmat : Rinv;
        var op = LPM.Kron(LPM.Kron(LPM.Eye(1 << i), local), LPM.Eye(1 << (n - i - 2)));
        output = op * output;
    }
    return output;
}

// Trace quantique : torsion par le Cartan K^(x)n, exposant = #(+1) - #(-1).
LP QuantumTrace(int n, LPM rho)
{
    var total = new LP();
    for (int col = 0; col < (1 << n); col++)
    {
        int e = 0;
        for (int z = 0; z < n; z++) e += ((col >> (n - 1 - z)) & 1) == 0 ? 1 : -1;
        total = total + rho.E[col, col].Shift(e);
    }
    return total;
}

// Polynome de Jones de la fermeture de `word`, en q (t = q^2 ensuite).
// Markowitz I : la trace quantique annule la dependance au choix de la tresse.
// Markov II  : la division par q + q^-1 et la correction q^(-2w) compensent le cadrage.
LP JonesPolynomial(int n, int[] word, int writhe)
{
    var rho = BraidOperator(n, word);
    var v = LP.DivExact(QuantumTrace(n, rho), qq + qm1);
    return v.Shift(-2 * writhe);
}

// Test de normalisation : le mot vide sur un brin est le noeud trivial, V = 1.
Console.WriteLine($"noeud trivial (mot vide, n = 1) : V = {JonesPolynomial(1, new int[0], 0)}   (attendu 1)");

Yang-Baxter R12 R23 R12 = R23 R12 R23 : True


noeud trivial (mot vide, n = 1) : V = 1   (attendu 1)


### Lecture du résultat

La chaîne tient en trois fonctions, et chacune traduit un théorème de Markov :

- `BraidOperator` réalise le **théorème d'Alexander** : tout nœud est la fermeture d'une tresse — ici, le trèfle est $\hat\sigma_1^3$, le nœud de huit $\widehat{\sigma_1\sigma_2^{-1}\sigma_1\sigma_2^{-1}}$, et $T(2,5)$ (le nœud torique à 5 croisements) est $\hat\sigma_1^5$ ;
- `QuantumTrace` réalise la **stabilité de Markov I** : la trace ordinaire ne convient pas (elle verrait la tresse, pas le nœud) ; la torsion par $K^{\otimes n}$ — l'action du Cartan de $U_q(\mathfrak{sl}_2)$ — annule exactement la dépendance au choix de la tresse ;
- la division par $[2]_q = q + q^{-1}$ et la correction $q^{-2w}$ réalisent la **stabilité de Markov II** : ajouter un tour de cadrage multiplie $V$ par $q^{\pm 2}$, que la correction compense.

La division exacte du §7 est l'endroit où une implémentation approximative tricherait sans le dire : `DivExact` lève si le reste n'est pas nul, donc un exposant faux ne produit pas un polynôme un peu différent — il produit une exception. Le test du nœud trivial, $V = 1$, confirme la normalisation.

In [15]:
// Les trois noeuds, contre les valeurs publiees (Knot Atlas).
// Le polynome est calcule en q, puis traduit en t = q^2 : pour un noeud, tous
// les exposants en q sont pairs, donc la substitution est exacte.
LP InT(LP p)
{
    var r = new LP();
    foreach (var kv in p.C)
    {
        if (((kv.Key % 2) + 2) % 2 != 0) throw new ArithmeticException("exposant impair en q");
        r.C[kv.Key / 2] = kv.Value;
    }
    return r;
}

var KNOTS = new (string Nom, int N, int[] Mot, int Enl)[]
{
    ("3_1 (trefle)",        2, new[] { 1, 1, 1 },          3),
    ("4_1 (noeud de huit)", 3, new[] { 1, -2, 1, -2 },     0),
    ("5_1 = T(2,5)",        2, new[] { 1, 1, 1, 1, 1 },    5),
};

// References du Knot Atlas, en t.
LP Ref(params (int E, int C)[] termes)
{
    var p = new LP();
    foreach (var (e, c) in termes) p = p + LP.Monom(e, c);
    return p;
}
var REFERENCE = new Dictionary<string, LP>
{
    ["3_1 (trefle)"]        = Ref((-4, -1), (-3, 1), (-1, 1)),
    ["4_1 (noeud de huit)"] = Ref((2, 1), (1, -1), (0, 1), (-1, -1), (-2, 1)),
    ["5_1 = T(2,5)"]        = Ref((-7, -1), (-6, 1), (-5, -1), (-4, 1), (-2, 1)),
};

Console.WriteLine(new string('=', 78));
Console.WriteLine($"{"noeud",-20} {"V_K(t) calcule",-40} accord");
Console.WriteLine(new string('=', 78));
bool tousOk = true;
foreach (var k in KNOTS)
{
    var v = InT(JonesPolynomial(k.N, k.Mot, k.Enl));
    bool ok = v.Same(REFERENCE[k.Nom]);
    tousOk = tousOk && ok;
    Console.WriteLine($"{k.Nom,-20} {v.ToString("t"),-40} {(ok ? "exact" : "ECART")}");
}
Console.WriteLine(new string('=', 78));
Console.WriteLine("Les trois polynomes coincident avec la reference : " + tousOk);

noeud                V_K(t) calcule                           accord


3_1 (trefle)         t^-1 + t^-3 - t^-4                       exact


4_1 (noeud de huit)  t^2 - t + 1 - t^-1 + t^-2                exact


5_1 = T(2,5)         t^-2 + t^-4 - t^-5 + t^-6 - t^-7         exact


Les trois polynomes coincident avec la reference : True


### Lecture du résultat

Trois nœuds, trois familles : le trèfle $\hat\sigma_1^3$ et $T(2,5) = \hat\sigma_1^5$ sont les premiers nœuds **toriques** (chiraux — le sens du mot de tresse fixe le sens d'enroulement), le nœud de huit est le premier nœud **amphichiral** (son polynôme est symétrique en $t \leftrightarrow t^{-1}$, il ne peut donc pas distinguer les deux miroirs).

L'accord est **exact, terme à terme**, sur les trois — obtenu par une voie qui ne partage aucune ligne avec la somme d'états de Kauffman ni avec Temperley–Lieb. Les symétries se lisent directement dans les exposants : $V(4_1)$ est palindromique (amphichiralité), $V(3_1)$ et $V(5_1)$ ne le sont pas (chiralité), et l'**envergure** de chaque polynôme — l'écart entre ses exposants extrêmes, $4 - 1 = 3$, $2 - (-2) = 4$, $-2 - (-7) = 5$ — égale le nombre de croisements : c'est le théorème de **Kauffman–Murasugi–Thistlethwaite** (l'égalité envergure = nombre de croisements vaut pour tout diagramme alterné réduit — nos trois nœuds le sont).

Le carnet Python obtenait les mêmes trois polynômes avec `sympy` ; ici `BigInteger` et la division exacte les obtiennent sans bibliothèque symbolique. Deux outils, un seul résultat — c'est la définition d'une vérification.

In [16]:
// Loi du miroir : renverser tous les croisements du mot revient a t -> 1/t.
Console.WriteLine("Loi du miroir : V_miroir(t) = V(t^-1)");
Console.WriteLine();
foreach (var (nom, longueur) in new[] { ("3_1", 3), ("5_1", 5) })
{
    var mot = Enumerable.Repeat(1, longueur).ToArray();
    var vPos = InT(JonesPolynomial(2, mot, longueur));
    var vNeg = InT(JonesPolynomial(2, mot.Select(s => -s).ToArray(), -longueur));
    bool ok = vNeg.Same(vPos.Reverse());
    Console.WriteLine($"  {nom} : V(sigma_1^{longueur})  = {vPos.ToString("t")}");
    Console.WriteLine($"        V(sigma_1^-{longueur}) = {vNeg.ToString("t")}");
    Console.WriteLine($"        loi du miroir verifiee : {ok}");
    Console.WriteLine();
}

Loi du miroir : V_miroir(t) = V(t^-1)


  3_1 : V(sigma_1^3)  = t^-1 + t^-3 - t^-4


        V(sigma_1^-3) = -t^4 + t^3 + t


        loi du miroir verifiee : True


  5_1 : V(sigma_1^5)  = t^-2 + t^-4 - t^-5 + t^-6 - t^-7


        V(sigma_1^-5) = -t^7 + t^6 - t^5 + t^4 + t^2


        loi du miroir verifiee : True


### Lecture du résultat

Le miroir du trèfle a un polynôme différent du trèfle ($-t^{-4}+t^{-3}+t^{-1}$ contre $-t^4+t^3+t$) : le trèfle est **chiral**, et l'invariant le voit. La substitution $t \to t^{-1}$ échange exactement les deux — c'est la loi du miroir, vérifiée ici sur les deux nœuds chiraux. Le nœud de huit, lui, vérifie la loi trivialement : son polynôme est invariant par la substitution, il ne distingue pas ses miroirs.

Physiquement : en Chern–Simons, passer au miroir revient à changer le signe du niveau $k \to -k$ (soit $q \to q^{-1}$). La chiralité détectée par l'invariant est donc la **parité** de la théorie topologique elle-même — Chern–Simons distingue un nœud de son image dans un miroir, ce qu'aucune théorie de jauge métrique ne fait.

In [17]:
// Specialisation Chern-Simons : au niveau k, la valeur physique est le polynome
// evalue en t_k = exp(2*pi*i/(k+2)) -- le decalage k+2 est la charge quantique.
Console.WriteLine(new string('=', 78));
Console.WriteLine("Specialisation Chern-Simons : t_k = exp(2*pi*i / (k+2))");
Console.WriteLine(new string('=', 78));
foreach (int k in new[] { 1, 2, 3, 4 })
{
    var tk = Complex.Exp(new Complex(0, 2 * Math.PI / (k + 2)));
    var cellules = new List<string>();
    foreach (var kn in KNOTS)
    {
        var v = InT(JonesPolynomial(kn.N, kn.Mot, kn.Enl));
        cellules.Add($"|V_{kn.Nom.Split(' ')[0]}| = {Complex.Abs(v.Eval(tk)):F6}");
    }
    Console.WriteLine($"  k={k} : " + string.Join("   ", cellules));
}
Console.WriteLine();
var enUn = new List<string>();
foreach (var kn in KNOTS)
{
    var v = InT(JonesPolynomial(kn.N, kn.Mot, kn.Enl)).Eval(Complex.One);
    enUn.Add($"{v.Real:F0}");
}
Console.WriteLine("V_K(1) (limite classique q -> 1) : [" + string.Join(", ", enUn) + "]  -- attendu : 1 pour tout noeud");

Specialisation Chern-Simons : t_k = exp(2*pi*i / (k+2))


  k=1 : |V_3_1| = 1.000000   |V_4_1| = 1.000000   |V_5_1| = 1.000000


  k=2 : |V_3_1| = 1.000000   |V_4_1| = 1.000000   |V_5_1| = 1.000000


  k=3 : |V_3_1| = 1.543362   |V_4_1| = 1.236068   |V_5_1| = 0.381966


  k=4 : |V_3_1| = 1.732051   |V_4_1| = 1.000000   |V_5_1| = 1.000000


V_K(1) (limite classique q -> 1) : [1, 1, 1]  -- attendu : 1 pour tout noeud


### Lecture du résultat — et ce que cette section **ne fait pas**

La spécialisation transforme un polynôme en un **nombre** par (nœud, niveau) : c'est la valeur du corrélateur de Wilson en Chern–Simons, c'est-à-dire une amplitude quantique de la théorie topologique. À $k = 1$ et $k = 2$, les trois nœuds valent 1 en module ; à $k = 3$ et $k = 4$, ils se distinguent — le niveau $k$ est bien un paramètre physique, pas une décoration. La limite classique $t \to 1$ redonne 1 pour tout nœud : c'est la normalisation du polynôme (et le seul point où les trois invariants coïncident nécessairement).

**Ce que cette section ne fait pas** : elle ne *déduit* pas le polynôme de Jones de l'intégrale de chemin de Chern–Simons — c'est le théorème de Witten (1989), cité et utilisé, pas redémontré. Ce qui est vérifié ici, c'est que la construction R-matrice + trace quantique **produit** l'invariant : trois nœuds dont les polynômes sont publiés, retrouvés exactement, et les deux lois de symétrie (miroir, limite classique) satisfaites. La voie sœur — crochet de Kauffman et Temperley–Lieb — est l'objet du carnet `KNOTS-02` ; les deux voies ne partagent aucune ligne de code.

### Ce que la division exacte garantit

Un détail qui n'en est pas un : la ligne `DivExact(QuantumTrace(...), q + q^-1)`. En flottant, cette division rendrait un polynôme « presque » juste, et une comparaison avec le Knot Atlas à tolérance près ne distinguerait pas une formule fausse d'une formule juste. En arithmétique exacte, il n'y a pas de tolérance : soit le reste est nul et le quotient est *le* polynôme, soit le reste ne l'est pas et le programme lève. Les trois accords `exact` de la cellule précédente sont donc des théorèmes vérifiés sur ces trois diagrammes, pas des mesures.

C'est le même principe qu'au §5 : la valeur d'une vérification est celle de son instrument. Là c'était une loi exacte de Bessel contre un Monte Carlo ; ici c'est $\mathbb{Z}[q,q^{-1}]$ contre des valeurs publiées.

### Ce que le jumeau C# ajoute — et ce qu'il ne change pas

Le carnet Python s'appuyait sur `sympy` : matrices symboliques, `factor`, `subs`. Le jumeau C# n'a **aucune** bibliothèque symbolique : il porte l'anneau $\mathbb{Z}[q,q^{-1}]$ lui-même, la multiplication de matrices $4\times4$ sur cet anneau, et la division exacte. Le résultat est identique — les trois polynômes, la loi du miroir, la table de Chern–Simons — mais la surface de vérification change : là où `sympy` pouvait masquer une erreur de convention derrière une simplification, ici chaque opération est visible.

C'est l'intérêt du jumeau : deux implémentations indépendantes de la même mathématique, comparées sur des valeurs publiées. Une erreur de signe dans la R-matrice, un exposant de trace quantique faux, un décalage d'enlacement oublié : chacun produirait un polynôme différent, et l'écart serait visible terme à terme.

### Le même invariant, deux lectures

Le §4 mesurait $W(C)$ sur réseau et lisait une décroissance en aire : une propriété **métrique**, donc un détecteur de confinement. Le §7 évalue la même boucle en Chern–Simons et obtient un **polynôme** : une propriété **topologique**, donc un invariant de nœud.

Ce n'est pas une contradiction : c'est le même objet dans deux théories différentes. L'action de Wilson dépend de la métrique (le pas $a$, les aires) ; l'action de Chern–Simons n'en dépend pas. Ce qui reste au corrélateur, une fois la métrique retirée, ne peut être qu'une classe d'isotopie — et c'est exactement ce que le polynôme de Jones mesure. Aucune des deux lectures ne se déduit de l'autre : c'est le point que la question 4 des exercices de compréhension fait formuler.

## §8. L'existence de Yang–Mills en dimension 4 : ce que le réseau peut en dire

Le carnet a mesuré une théorie de jauge **sur réseau**. Le problème du millénaire demande autre chose : l'**existence** d'une théorie quantique des champs de Yang–Mills en dimension 4, avec un **gap de masse** — et c'est un énoncé sur le **continu**, pas sur le réseau.

Le 28 octobre 2026, une soumission formelle a été publiée : le dépôt Lean [`yang-mills-group/OpenAI-YM-20261028`](https://github.com/yang-mills-group/OpenAI-YM-20261028) (licence Apache-2.0), dont le théorème unique est `YangMills.jointExistence`. Ce n'est pas un théorème isolé mais une **existence conjointe** : pour tout groupe de jauge compact d'algèbre de Lie réelle simple non abélienne, il existe une famille de Wilson sur réseau *et* une hiérarchie de continuum, et **quatre** propriétés sont satisfaites **simultanément** par cette même hiérarchie.

| Propriété | Ce qu'elle exige |
|---|---|
| Origine de Wilson | une pente positive $\beta/\log(\text{maille}^{-1})$ fixée *avant* le choix de la famille, des insertions invariantes de jauge à support local fini, et la limite « maillage d'abord, volume ensuite » |
| Reconstruction de Wightman | les axiomes W0–W3, une mesure de phase normalisée à support plein, la symétrie de permutation, la lissabilité, et la continuité entre l'axe imaginaire et les moments de Wightman |
| OPE séparée à tous les ordres | un système **unique** de coefficients, valable à chaque degré de troncature et dans chaque phase |
| Test de non-liberté | la fonction à deux points de la trace centrée d'ordre 4 vérifie $r^{8}\log^{2}(1/r)\,G(r) \to 3\,\dim(\mathfrak{g})/\pi^{4} > 0$, et **aucune** somme finie de profils de Wick mass-shell ne peut lui être égale au voisinage de $0$ |

Deux avertissements, qui font partie de la soumission elle-même : le **gap de masse** est **hors périmètre** (la soumission ne résout donc pas le problème Clay), et l'**unicité** n'est démontrée que pour le **vide** de Wightman, pas pour la prescription de champ.

Ce qui reste calculable sans Lean, c'est le **dernier** point de la table : le test de non-liberté est arithmétique. On le refait ici, et on vérifie que la famille des profils libres est **incapable** d'atteindre le coefficient exigé.

In [18]:
// Fonction a deux points d'un champ libre scalaire de masse m, en temps egal euclidien 4D :
//     G_m(r) = int d^3p/(2 pi)^3 * 1/(2 E_p) e^{-r E_p},   E_p = sqrt(p^2 + m^2).
// Deux voies independantes :
//   (a) forme close  G_m(r) = m^2/(8 pi^2) (K_2(m r) - K_0(m r)) ;
//   (b) quadrature de l'integrale en p, variable t = r p.

// Noeuds et poids de Gauss-Legendre sur [-1, 1] par Newton sur P_n (BCL seule).
(double[] X, double[] W) GaussLegendre(int n)
{
    var x = new double[n];
    var w = new double[n];
    for (int i = 0; i < n; i++)
    {
        double z = Math.Cos(Math.PI * (i + 0.75) / (n + 0.5));
        for (int it = 0; it < 100; it++)
        {
            double p0 = 1.0, p1 = z;
            for (int kk = 2; kk <= n; kk++)
            {
                double p2 = ((2 * kk - 1) * z * p1 - (kk - 1) * p0) / kk;
                p0 = p1; p1 = p2;
            }
            double dp = n * (z * p1 - p0) / (z * z - 1);
            double dz = p1 / dp;
            z -= dz;
            if (Math.Abs(dz) < 1e-16) break;
        }
        double P0 = 1.0, P1 = z;
        for (int kk = 2; kk <= n; kk++)
        {
            double P2 = ((2 * kk - 1) * z * P1 - (kk - 1) * P0) / kk;
            P0 = P1; P1 = P2;
        }
        double dP = n * (z * P1 - P0) / (z * z - 1);
        x[i] = z;
        w[i] = 2.0 / ((1 - z * z) * dP * dP);
    }
    return (x, w);
}

var (xn, wn) = GaussLegendre(40);

// Quadrature composite : panneaux geometriques pres de 0, ou la singularite logarithmique de K_0 vit.
double Integre(Func<double, double> f, double[] bornes)
{
    double s = 0;
    for (int p = 0; p + 1 < bornes.Length; p++)
    {
        double a = bornes[p], b = bornes[p + 1];
        double mid = 0.5 * (a + b), half = 0.5 * (b - a);
        for (int k = 0; k < xn.Length; k++) s += wn[k] * f(mid + half * xn[k]) * half;
    }
    return s;
}

double[] PANELS = { 0.0, 1e-4, 1e-3, 1e-2, 0.05, 0.2, 0.5, 1.0, 2.0, 4.0, 8.0, 12.0 };

// K_nu(x) = 2/sqrt(x) e^-x int_0^inf e^{-u^2} T_nu(1 + u^2/x) / sqrt(2 + u^2/x) du.
// Identite obtenue par le changement de variable s = sinh t, w = cosh t - 1 = v^2 ;
// T_nu est le polynome de Tchebychev de premiere espece (T_0 = 1, T_2 = 2z^2 - 1).
double BesselK(int nu, double x)
{
    double c = 1.0 / x;
    double F(double u)
    {
        double z = 1 + u * u * c;
        double T = nu == 0 ? 1.0 : 2 * z * z - 1;
        return Math.Exp(-u * u) * T / Math.Sqrt(2 + u * u * c);
    }
    return 2 / Math.Sqrt(x) * Math.Exp(-x) * Integre(F, PANELS);
}

double GLibreFermee(double r, double m)
{
    double z = m * r;
    return m * m / (8 * Math.PI * Math.PI) * (BesselK(2, z) - BesselK(0, z));
}

double[] PP = { 0.0, 0.2, 0.5, 1.0, 2.0, 4.0, 8.0, 15.0, 30.0 };

double GLibreQuadrature(double r, double m)
{
    double a = m * r;
    double F(double t) => t * t / Math.Sqrt(t * t + a * a) * Math.Exp(-Math.Sqrt(t * t + a * a));
    return Integre(F, PP) / (4 * Math.PI * Math.PI * r * r);
}

var grille = new (double R, double M)[] { (0.05, 0.5), (0.1, 1.0), (0.25, 0.1), (0.5, 2.0) };
double ecartMax = 0;
foreach (var (r, m) in grille)
{
    double a1 = GLibreFermee(r, m), b1 = GLibreQuadrature(r, m);
    ecartMax = Math.Max(ecartMax, Math.Abs(a1 - b1) / Math.Abs(a1));
}
Console.WriteLine($"ecart relatif maximal entre les deux voies : {ecartMax:E3}");
Console.WriteLine($"valeur en (r=0.1, m=1.0) : {GLibreFermee(0.1, 1.0):F10}");

// Troisieme controle, sans Bessel : la limite de masse nulle G_0(r) = 1/(4 pi^2 r^2).
Console.WriteLine();
Console.WriteLine("limite de masse nulle (controle sans Bessel) :");
foreach (double r in new[] { 0.1, 0.25 })
{
    double mesure = Integre(t => t * Math.Exp(-t), PP) / (4 * Math.PI * Math.PI * r * r);
    double attendu = 1 / (4 * Math.PI * Math.PI * r * r);
    Console.WriteLine($"  r={r} : {mesure:F10} | attendu 1/(4 pi^2 r^2) = {attendu:F10}");
}

ecart relatif maximal entre les deux voies : 4.740E-012


valeur en (r=0.1, m=1.0) : 2.4960080416


limite de masse nulle (controle sans Bessel) :


  r=0.1 : 2.5330295911 | attendu 1/(4 pi^2 r^2) = 2.5330295911


  r=0.25 : 0.4052847346 | attendu 1/(4 pi^2 r^2) = 0.4052847346


### Lecture du résultat

Les deux voies de calcul — forme close en Bessel et quadrature directe de l'intégrale — coïncident à mieux que $10^{-11}$ en relatif sur la grille. Un troisième contrôle, sans Bessel, valide la **normalisation** : la limite de masse nulle redonne $1/(4\pi^{2}r^{2})$, la formule connue du propagateur libre euclidien 4D à temps égal. On peut donc traiter $G_m$ comme une fonction de référence fiable.

C'est la partie du carnet où la bibliothèque standard est la plus sollicitée, et elle suffit : 40 nœuds de Gauss–Legendre par panneau, des panneaux géométriques près de zéro — là où la singularité logarithmique de $K_0$ vit —, et l'identité intégrale pour $K_\nu$, que donne le changement de variable $s = \sinh t$ (voir les commentaires de la cellule de code). Le carnet Python mène les mêmes deux voies avec `mpmath` à 40 chiffres et trouve un écart de l'ordre de $10^{-41}$ ; ici le `double` de la bibliothèque standard trouve $5 \times 10^{-12}$, ce qui suffit à la lecture à quatre chiffres significatifs qui suit. La différence d'instrument est déclarée, la conclusion est la même.

Reste l'opération qui porte tout le test : la **renormalisation** $r^{8}\log^{2}(1/r)\,G(r)$. C'est elle qui isole le coefficient ultraviolet.

In [19]:
// La renormalisation imposee par le test de non-liberte.
double Renormalise(Func<double, double> f, double r)
    => Math.Pow(r, 8) * Math.Pow(Math.Log(1 / r), 2) * f(r);

Console.WriteLine("R(r) = r^8 log^2(1/r) * G_m(r)   (profils libres)");
foreach (double m in new[] { 0.1, 0.5, 1.0, 2.0 })
{
    var morceaux = new List<string>();
    foreach (double r in new[] { 1e-2, 1e-3, 1e-4, 1e-5 })
        morceaux.Add($"r={r:E0} -> {Renormalise(x => GLibreFermee(x, m), r):E4}");
    Console.WriteLine($"  m = {m} : " + string.Join("   ", morceaux));
}

// Loi d'echelle attendue si le terme dominant est 1/(4 pi^2 r^2).
Console.WriteLine();
foreach (double r in new[] { 1e-3, 1e-4, 1e-5 })
{
    double prevu = Math.Pow(r, 6) * Math.Pow(Math.Log(1 / r), 2) / (4 * Math.PI * Math.PI);
    double mesure = Renormalise(x => GLibreFermee(x, 1.0), r);
    Console.WriteLine($"  r={r:E0} : mesure {mesure:E5} | prevu r^6 log^2(1/r)/(4 pi^2) {prevu:E5}");
}

R(r) = r^8 log^2(1/r) * G_m(r)   (profils libres)


  m = 0.1 : r=1E-002 -> 5.3719E-013   r=1E-003 -> 1.2087E-018   r=1E-004 -> 2.1488E-024   r=1E-005 -> 3.3575E-030


  m = 0.5 : r=1E-002 -> 5.3715E-013   r=1E-003 -> 1.2087E-018   r=1E-004 -> 2.1488E-024   r=1E-005 -> 3.3575E-030


  m = 1 : r=1E-002 -> 5.3705E-013   r=1E-003 -> 1.2087E-018   r=1E-004 -> 2.1488E-024   r=1E-005 -> 3.3575E-030


  m = 2 : r=1E-002 -> 5.3671E-013   r=1E-003 -> 1.2087E-018   r=1E-004 -> 2.1488E-024   r=1E-005 -> 3.3575E-030


  r=1E-003 : mesure 1.20868E-018 | prevu r^6 log^2(1/r)/(4 pi^2) 1.20869E-018


  r=1E-004 : mesure 2.14878E-024 | prevu r^6 log^2(1/r)/(4 pi^2) 2.14878E-024


  r=1E-005 : mesure 3.35747E-030 | prevu r^6 log^2(1/r)/(4 pi^2) 3.35747E-030


### Lecture du résultat

Le résultat **ne dépend pas de $m$** et tend vers **zéro** comme $r^{6}\log^{2}(1/r)$ : la singularité $1/r^{2}$ du propagateur libre à temps égal domine, et le facteur de renormalisation la repousse à zéro. Les quatre masses donnent la même colonne à quatre chiffres — c'est la signature du terme dominant universel.

C'est exactement ce que le test de non-liberté exploite : si **toute** fonction à deux points libre renormalisée s'annule, alors un coefficient **fini et non nul** après la même renormalisation ne peut pas provenir d'une somme de profils de Wick. La cellule suivante mesure les deux côtés de cette alternative.

In [20]:
// La cible imposee par le test de non-liberte, pour SU(2) : dim_R(su(2)) = 3.
int dimSu2 = 3;
double cible = 3.0 * dimSu2 / Math.Pow(Math.PI, 4);
Console.WriteLine($"coefficient exige : 3 * dim(g) / pi^4 = {cible:F8}");

// La clause d'exclusion porte sur TOUTE somme finie de profils de Wick mass-shell.
// On echantillonne une famille (masses et coefficients complexes arbitraires).
var rngYm = new Random(7);
var massesJeu = new List<double[]>();
var coeffsJeu = new List<Complex[]>();
for (int k = 0; k < 6; k++)
{
    var ms = new double[3];
    var cs = new Complex[3];
    for (int j = 0; j < 3; j++)
    {
        ms[j] = 0.05 + rngYm.NextDouble() * (3.0 - 0.05);
        cs[j] = new Complex(Gauss(rngYm), Gauss(rngYm));
    }
    massesJeu.Add(ms);
    coeffsJeu.Add(cs);
}

// La renormalisation est lineaire : renormaliser chaque profil puis combiner
// garde la partie imaginaire des coefficients, au lieu de la perdre en ne
// gardant que la partie reelle de la somme.
Complex RenormaliseCompose(double[] ms, Complex[] cs)
{
    var s = Complex.Zero;
    for (int j = 0; j < ms.Length; j++)
        s += cs[j] * Renormalise(x => GLibreFermee(x, ms[j]), 1e-4);
    return s;
}

var mesures = new List<Complex>();
for (int k = 0; k < 6; k++)
{
    var ms = massesJeu[k];
    var cs = coeffsJeu[k];
    mesures.Add(RenormaliseCompose(ms, cs));
}

Console.WriteLine();
Console.WriteLine("limite renormalisee des sommes finies de profils libres :");
for (int k = 0; k < mesures.Count; k++)
    Console.WriteLine($"  combinaison {k + 1} : {mesures[k].Real:E6} {mesures[k].Imaginary:+0.000000E+00;-0.000000E+00}   (cible : {cible:F6})");
double ecartMin = mesures.Min(v => Complex.Abs(v - cible));
Console.WriteLine();
Console.WriteLine($"ecart minimal a la cible : {ecartMin:F6}");

coefficient exige : 3 * dim(g) / pi^4 = 0.09239384


limite renormalisee des sommes finies de profils libres :


  combinaison 1 : -6.393158E-024 +2.835349E-24   (cible : 0.092394)


  combinaison 2 : 4.513290E-024 +1.930918E-24   (cible : 0.092394)


  combinaison 3 : 5.966194E-024 +2.896342E-24   (cible : 0.092394)


  combinaison 4 : -2.930000E-024 -3.129341E-24   (cible : 0.092394)


  combinaison 5 : -3.207696E-025 +4.201927E-24   (cible : 0.092394)


  combinaison 6 : 3.442620E-024 -6.236164E-24   (cible : 0.092394)


ecart minimal a la cible : 0.092394


### Lecture du résultat

Tous les profils libres renormalisés tendent vers zéro, quelle que soit la masse et quelle que soit la combinaison linéaire complexe — y compris les sommes finies, que la clause d'exclusion de l'énoncé met hors jeu. L'écart à la cible $9/\pi^{4}$ reste donc de l'ordre de la cible elle-même : la famille libre **ne peut pas** atteindre le coefficient exigé.

Note : la combinaison est complexe, et ses **deux** parties sont affichées. Elles tendent l'une et l'autre vers zéro ; le module, qui est la quantité comparée à la cible, reste de l'ordre de $10^{-24}$ — l'écart à $9/\pi^{4}$ vaut la cible elle-même.

Reste à montrer que le test n'est pas **vide** : il faut exhiber une fonction à deux points, non libre par construction, qui atteint la cible.

In [21]:
// Le test n'est pas vide : il existe des fonctions a deux points dont la limite
// renormalisee vaut exactement le coefficient exige. Modele de trace, non libre.
double GTraceModele(double r, double c) => c / (Math.Pow(r, 8) * Math.Pow(Math.Log(1 / r), 2)) + GLibreFermee(r, 1.0);

Console.WriteLine("limite renormalisee du modele de trace (non libre) :");
foreach (double r in new[] { 1e-2, 1e-3, 1e-4, 1e-5 })
{
    double valeur = Renormalise(x => GTraceModele(x, cible), r);
    Console.WriteLine($"  r={r:E0} : {valeur:F10}   (cible : {cible:F10})");
}

limite renormalisee du modele de trace (non libre) :


  r=1E-002 : 0.0923938403   (cible : 0.0923938403)


  r=1E-003 : 0.0923938403   (cible : 0.0923938403)


  r=1E-004 : 0.0923938403   (cible : 0.0923938403)


  r=1E-005 : 0.0923938403   (cible : 0.0923938403)


### Lecture du résultat

Trois faits, et ils se tiennent :

1. **Tous les profils libres renormalisés tendent vers zéro**, quelle que soit la masse, et quelle que soit la combinaison linéaire complexe. L'écart à la cible $9/\pi^{4}$ reste de l'ordre de la cible elle-même : la famille libre **ne peut pas** atteindre le coefficient exigé.
2. **La cible est atteinte par autre chose.** Le modèle de trace construit ici (non libre, par construction) converge vers $3\dim(\mathfrak{g})/\pi^{4}$ exactement — le terme $c/(r^{8}\log^{2}(1/r))$ domine, et la renormalisation l'isole.
3. **La loi d'échelle est la signature de l'interaction.** Un profil libre renormalisé décroît comme $r^{6}\log^{2}(1/r)$ — la trace de la singularité $1/r^{2}$ du propagateur libre. Un coefficient *fini* et *non nul* après la même renormalisation ne peut pas provenir d'une somme de Wick : c'est exactement ce que l'énoncé appelle « non-liberté ».

### Ce que cette section ne fait pas

Elle **illustre et vérifie la partie calculable** d'un énoncé formel, elle ne le prouve pas. La compilation Lean de la soumission n'est **pas** rejouée ici (la compilation est du ressort de la CI), et la lecture exhaustive de ses sources est consignée à part dans [`docs/research/yang-mills-openai-20261028-analyse.md`](../../../docs/research/yang-mills-openai-20261028-analyse.md), avec un outil de relecture rejouable ([`scripts/ym_openai_20261028/inventory.py`](../../../scripts/ym_openai_20261028/inventory.py)).

## §9. Synthèse

| Question posée | Réponse de ce carnet | Instrument | Verdict |
|---|---|---|---|
| L'invariance de jauge survit-elle à la discrétisation ? | oui, **exactement** à tout pas $a$ | plaquette $SU(2)$, action $U(1)$ sur $3^4$ | traces et action invariantes à la précision machine |
| Le continu peut-il voir le confinement ? | non — propriété **non perturbative** | boucle de Wilson $W(C)$ | loi d'aire validée contre la loi exacte 2D ($\sigma = -\ln(I_1/I_0)$) |
| Une observable peut-elle trier les phases ? | oui : $\chi(1,1)$ chute d'un facteur plusieurs entre les deux phases | Monte Carlo 4D, $\beta = 0{,}6$ contre $1{,}2$ | bascule détectée (rapport $4{,}30$), cohérente avec le $\beta_c \approx 1{,}01$ connu par ailleurs |
| La même observable devient-elle un invariant de nœud en théorie topologique ? | oui : en Chern–Simons $SU(2)$ au niveau $k$, $\langle W(K)\rangle$ est le polynôme de Jones en $t = e^{2\pi i/(k+2)}$ | R-matrice $U_q(\mathfrak{sl}_2)$ + trace quantique (§7) | accord **exact** avec le Knot Atlas sur $3_1$, $4_1$, $5_1$, par une voie indépendante de Temperley–Lieb |
| Peut-on dire quelque chose de l'existence Yang–Mills ? | oui, mais **pas le gap de masse** : existence d'une hiérarchie constructible + un test de non-liberté au coefficient $3\dim(\mathfrak{g})/\pi^{4}$ | renormalisation $r^{8}\log^{2}(1/r)$ des profils de Wick (§8) | tous les profils libres renormalisés vont à $0$, jamais à $9/\pi^{4}$ ; la cible est atteignable par un modèle non libre |

Chaque étage du carnet a été ancré à une **référence fermée ou établie** : le $\Delta S$ des agrafes contre l'action recomptée, la tension de ficelle contre la loi de Bessel exacte, la bascule contre le $\beta_c$ établi par ailleurs — le seul étage dont ce carnet ne fait pas la démonstration — et les trois polynômes de Jones contre les valeurs publiées du Knot Atlas. Le chemin parcouru sur la série : Noether (conservation) $\to$ invariance locale (dérivée covariante, $F_{\mu\nu}$) $\to$ **discrétisation exacte** (liens, plaquettes, action de Wilson) $\to$ observable non perturbative (boucle de Wilson, confinement) $\to$ la **même observable** en théorie topologique (Chern–Simons, invariant de nœud).

**Ce que le jumeau C# change, et ce qu'il ne change pas.** Les valeurs stochastiques diffèrent du carnet Python — `System.Random` n'est pas PCG64 — mais aucune conclusion n'en dépend : les cibles sont des lois exactes, les seuils sont pré-enregistrés, et les invariants du §7 sont exacts par construction. L'instrument change, les verdicts sont les mêmes : c'est ce qu'un jumeau doit démontrer.

## §10. Exercices

Quatre exercices : les trois premiers s'approprient le simulateur, le quatrième la voie Chern–Simons du §7. Les cellules s'exécutent sans erreur : à vous d'écrire le corps.

### Exercice 1 — loi de périmètre contre loi d'aire

Les rectangles $2 \times 2$ et $1 \times 4$ ont la même **aire** (4) mais des **périmètres** différents (8 contre 10). Si c'est l'aire qui fixe le coût d'une boucle, les deux $\langle W\rangle$ doivent coïncider à l'erreur statistique près ; si c'est le périmètre, ils doivent différer.

Étendre `LoopAngle` en une mesure de contour quelconque, mesurer les deux familles à $\beta = 1$, et conclure.

**Indication** — un contour quelconque est une liste de segments `(direction, longueur, signe)` ; accumuler les phases comme dans `LoopAngle`.

**Étapes**
1. Généraliser `LoopAngle` en `LoopAngleContour(th, lat, segments)`.
2. Mesurer les deux contours de même aire sur un même run.
3. Comparer et conclure sur la loi qui gagne.

In [22]:
// Exercice 1 -- loi de perimetre contre loi d'aire.
// Les rectangles 2x2 et 1x4 ont la meme aire (4) mais des perimetres
// differents (8 contre 10). Etendre LoopAngle en une mesure de contour
// quelconque, mesurer <W> pour les deux familles a beta = 1.0, et verifier que
// les deux valeurs coincident a l'erreur pres : c'est l'AIRE qui fixe le cout.
// Indication : un contour quelconque = liste de segments (direction, longueur, signe) ;
// accumuler les phases comme dans LoopAngle.
// Etape 1 : generaliser LoopAngle en LoopAngleContour(th, lat, segments).
// Etape 2 : mesurer les deux contours de meme aire sur un meme run.
// Etape 3 : comparer et conclure sur la loi qui gagne.
Func<bool> resultatEx1 = null;   // TODO etudiant

Console.WriteLine("Exercice a completer : loi de perimetre contre loi d aire");

Exercice a completer : loi de perimetre contre loi d aire


### Exercice 2 — passage à $SU(2)$ : tension de ficelle en couplage fort

Le simulateur du carnet est $U(1)$. Adapter la mise à jour à des liens $SU(2)$, avec des propositions $U' = e^{\epsilon\, i\, \vec{n}\cdot\vec{\sigma}}\, U$ où $\vec{n}$ est un vecteur unité aléatoire, et l'action $S = -\frac{\beta}{2}\mathrm{Re}\,\mathrm{Tr}\,P$.

En dimension 2 à $\beta = 0{,}6$, le développement en couplage fort prédit $\langle W(A)\rangle \sim (\beta/4)^{A}$. Mesurer $\chi(1,1) = -\ln\langle W(1,1)\rangle$, comparer à $-\ln(\beta/4)$, et discuter l'écart (corrections $O(\beta^{2})$).

**Indication** — si $P$ est paramétré par un quaternion $q$, $\mathrm{Tr}\,P = 2\,\mathrm{Re}(q_0)$ ; la formule d'agrafe reste valable, seule la mesure change.

In [23]:
// Exercice 2 -- passage a SU(2) : tension de ficelle en couplage fort.
// Adapter le simulateur a des liens SU(2) : propositions U' = exp(eps i n.sigma) U
// avec n aleatoire uniforme, action -beta/2 Re Tr P (N = 2). En dimension 2 a
// beta = 0.6, le developpement en couplage fort predit <W(A)> ~ (beta/4)^A.
// Mesurer chi(1,1) et comparer a -ln(beta/4) ; discuter l ecart (corrections O(beta^2)).
// Indication : pour les traces, Tr P = 2 Re(q0) si P est parametre par quaternion q.
Func<bool> resultatEx2 = null;   // TODO etudiant

Console.WriteLine("Exercice a completer : SU(2) et tension de ficelle");

Exercice a completer : SU(2) et tension de ficelle


### Exercice 3 — rapport de Creutz $\chi(2,2)$ et corrections au couplage fort

Mesurer $W(2,2)$, $W(2,1)$, $W(1,2)$ et $W(1,1)$ en dimension 2 à $\beta = 0{,}8$, puis former

$$\chi(2,2) = -\ln \frac{W(2,2)\,W(1,1)}{W(2,1)\,W(1,2)}.$$

Ce rapport **annule les contributions de périmètre** et isole la tension d'aire. Le comparer à $\sigma_{\mathrm{exact}}(0{,}8)$ : à ce $\beta$, les corrections $O(\beta^{2})$ à la loi d'aire deviennent sensibles — les chiffrer.

**Indication** — `LoopAngle` mesure déjà des rectangles $R \times T$ quelconques ; `SimulateU1` accepte une liste de formes.

In [24]:
// Exercice 3 -- rapport de Creutz chi(2,2) et corrections au couplage fort.
// Mesurer W(2,2), W(2,1), W(1,2) et W(1,1) en 2D a beta = 0.8 (rectangles R x T),
// puis former chi(2,2) = -ln[ W(2,2) W(1,1) / (W(2,1) W(1,2)) ] : ce rapport
// annule les contributions de perimetre et isole la tension d aire.
// Comparer a sigma_exact(0.8) : a ce beta, les corrections O(beta^2) a la loi
// d aire deviennent sensibles -- les chiffrer.
// Indication : LoopAngle mesure deja des rectangles R x T quelconques.
Func<bool> resultatEx3 = null;   // TODO etudiant

Console.WriteLine("Exercice a completer : rapport de Creutz chi(2,2)");

Exercice a completer : rapport de Creutz chi(2,2)


### Exercice 4 — $T(2,7)$ par la R-matrice, et la forme fermée des nœuds toriques

Avec `JonesPolynomial` (§7), calculer $V$ de la fermeture de $\sigma_1^{7}$, puis :

**Étape 1** — vérifier l'**envergure** du polynôme (attendu : 7). C'est l'égalité de Kauffman–Murasugi–Thistlethwaite, vraie pour tout diagramme alterné réduit.

**Étape 2** — vérifier la forme fermée des nœuds toriques $T(2,n)$, $n$ impair :

$$V_{T(2,n)}(t) = t^{-(3n-1)/2}\left(t^{n} + t^{n-2} + \sum_{j=0}^{n-3}(-1)^{j+1} t^{j}\right)$$

contre le calcul direct pour $n = 3, 5, 7$.

**Indication** — le mot est `[1] * n` et l'enlacement vaut $n$ ; l'envergure se lit sur `MinE` et `MaxE` de l'`LP` en $t$, la forme fermée se construit terme à terme avec `LP.Monom`.

In [25]:
// Exercice 4 -- T(2,7) par la R-matrice, et la forme fermee des noeuds toriques.
// TODO etudiant : avec JonesPolynomial (section 7), calculer V de la fermeture de sigma_1^7,
// puis :
// Etape 1 : verifier l'envergure du polynome (attendu : 7 ; c'est l'egalite de
// Kauffman-Murasugi-Thistlethwaite, vraie pour tout diagramme alterné reduit) ;
// Etape 2 : verifier la forme fermee des noeuds toriques T(2,n), n impair :
//     V_{T(2,n)}(t) = t^{-(3n-1)/2} * ( t^n + t^{n-2} + somme_{j=0}^{n-3} (-1)^{j+1} t^j )
//     contre le calcul direct pour n = 3, 5, 7.
// Indice : le mot est [1] * n et l'enlacement vaut n ; les exposants se lisent
// sur MinE et MaxE de l'LP en t, la forme fermee se construit avec LP.Monom.
int? resultatEx4Envergure = null;      // TODO etudiant : l'envergure de V_{T(2,7)} (un entier)
bool? resultatEx4FormeFermee = null;   // TODO etudiant : true si la forme fermee coincide pour n = 3, 5, 7

Console.WriteLine("Exercice a completer");

Exercice a completer


## À retenir

| Notion | Ce qu'elle dit | Où la voir |
|---|---|---|
| Transport parallèle | les liens portent $U \in SU(2)$, la plaquette $P = UUUU$ et $\mathrm{Tr}\,P$ sont invariants de jauge locale : écart $2{,}2\times 10^{-16}$, résidu de factorisation $2{,}5\times 10^{-16}$ | §1, §2 |
| Action exactement invariante | $\lvert\Delta S_W\rvert$ nul sur le tirage du carnet, d'au plus un ULP sur 92 % de 400 tirages de contrôle — le **télescopage** : les $\Omega$ d'une transformation locale se compensent plaquette à plaquette ; les agrafes prédisent $\Delta S$ à quelques $10^{-15}$ près | §3, §5 |
| Dimension 2 exacte | $\sigma_{\text{exact}}(0{,}6) = -\ln(I_1/I_0) = 1{,}2474$ ; mesuré $\sigma = 1{,}2514$, écart $0{,}33\ \%$ (seuil $5\ \%$) — hypothèse **pré-enregistrée avant la mesure** : CONFIRMÉE | §5 |
| Loi d'aire | $\ln W$ proportionnel à l'aire : pente ajustée $-0{,}805$ contre prédiction exacte $-\sigma(1{,}0) = -0{,}807$ ($A = 1, 2, 3$) | §5 |
| Contre-témoin 4D | $\chi(1{,}1) = 1{,}2236$ à $\beta = 0{,}6$ contre $0{,}2848$ à $\beta = 1{,}2$ (rapport $4{,}297$) — critères pré-enregistrés ($> 3$, $< 0{,}35$, $> 0{,}8$) : **bascule de phase détectée** | §6 |
| Boucle de Wilson topologique | en Chern–Simons $SU(2)_k$, la même observable vaut le polynôme de Jones : R-matrice $U_q(\mathfrak{sl}_2)$ (Hecke + Yang–Baxter exacts) et trace quantique donnent $V$ exact sur $3_1$, $4_1$, $5_1$ — puis un **nombre** à chaque niveau $k$ | §7 |
| Non-liberté | la trace centrée d'ordre 4 a pour limite renormalisée $r^{8}\log^{2}(1/r)\,G(r) \to 3\,\dim(\mathfrak{g})/\pi^{4} = 9/\pi^{4} \approx 0{,}0924$ pour $SU(2)$, alors que **toute** somme finie de profils de Wick mass-shell renormalisée tend vers $0$ | §8 |
| Existence (périmètre) | la soumission formelle établit l'existence d'une hiérarchie Yang–Mills 4D constructible — **gap de masse et unicité de la prescription hors périmètre** ; ce n'est pas le problème Clay | §8 |

La boucle ouverte en Jauge-01 est refermée : la symétrie qui produisait une conservation produit maintenant, posée sur réseau, une observable qui **discrimine les phases** de la théorie — confiner ses charges sans jamais casser l'invariance — et, poussée en théorie topologique, un **invariant de nœud**. La suite de la série quittera la mesure pour la **formulation** : développement en caractères et dualité exacte des modèles de jauge abéliens en dimension 2 (gaz de Coulomb, gaz de vortex).

## Vérifiez votre compréhension

1. La cellule du §3 affiche $|\Delta S_W| = 0$, exactement. Pourquoi ce zéro est-il normal alors que la transformation change presque tous les liens, et qu'est-ce qui garantit l'invariance ?

<details><summary>Réponse</summary>

Ce qui garantit l'invariance, c'est le **télescopage** : dans une transformation de jauge locale, chaque $\Omega$ apparaît une fois dans une plaquette et une fois dans la plaquette voisine, en positions conjuguées ; pour $U(1)$, les phases $\alpha$ des quatre sites d'un plaquette se compensent de même. Le zéro, lui, est un effet d'arrondi : la transformation change 309 des 486 angles de plaquette, mais chaque terme $1-\cos\theta$ recalculé ne diffère de l'ancien que de $3{,}3\times 10^{-15}$ au plus, bien moins que l'ULP de la somme ($5{,}7\times 10^{-14}$), et les additions arrondies effacent ces écarts. Sur 400 autres tirages, le résidu est exactement nul 210 fois et vaut au plus un ULP dans 92 % des cas. Les autres échelons de la chaîne disent la même chose : $\mathrm{Tr}\,P$ invariante à $2{,}2\times 10^{-16}$ près, résidu de factorisation $2{,}5\times 10^{-16}$, agrafes et action recomptée d'accord à quelques $10^{-15}$ près. Un défaut se verrait d'emblée : un lien mal transformé changerait l'action d'une quantité d'ordre $\beta$.

</details>

2. Que teste l'hypothèse pré-enregistrée de la dimension 2, et que changerait-il de l'écrire *après* la mesure ?

<details><summary>Réponse</summary>

Elle fixe **avant toute mesure** la cible ($\sigma_{\text{exact}}(0{,}6) = -\ln(I_1/I_0) = 1{,}2474$, contre le témoin $-\ln(\beta/2) = 1{,}2040$), le protocole (réseau $12 \times 12$, $\beta = 0{,}6$, 300 thermalisations + 6000 mesures) et le seuil ($5\ \%$). Écrite après, elle ne testerait plus rien : on aurait pu choisir la cible la plus proche du résultat — le témoin $1{,}2040$ n'est qu'à $3{,}5\ \%$, et la loi exacte discrimine là où il échouerait.

</details>

3. Le test 2D valide une loi exacte ; le contre-témoin 4D montre autre chose. Quoi, et par quels chiffres ?

<details><summary>Réponse</summary>

Une **transition de phase** : en 4D à couplage fort ($\beta = 0{,}6$), $\chi(1{,}1)$ est d'ordre unitaire (confinement) ; à $\beta = 1{,}2$, bien au-delà de $\beta_c \approx 1{,}01$, il s'effondre (phase de Coulomb). Les critères pré-enregistrés ($\chi(0{,}6)/\chi(1{,}2) > 3$, $\chi(1{,}2) < 0{,}35$, $\chi(0{,}6) > 0{,}8$) sont satisfaits : **bascule détectée**. Là où le 2D confronte le réseau à un calcul exact, le 4D confronte le réseau à la physique — aucune loi exacte n'y existe, seule la signature de la transition.

</details>

4. Le §4 mesure $W(C)$ sur réseau pour détecter le confinement ; le §7 évalue la même boucle en Chern–Simons et obtient un polynôme. Pourquoi le même observable donne-t-il des réponses physiques aussi différentes ?

<details><summary>Réponse</summary>

Parce que la **théorie** a changé, pas l'observable. Sur réseau, l'action de Wilson dépend de la métrique (le pas $a$, les aires) : $W(C) \sim e^{-\sigma A}$ décroît avec la **surface** parce que $A$ est une grandeur métrique — c'est ce qui fait de $\sigma$ une tension de ficelle, donc un détecteur de confinement. En Chern–Simons, l'action est **topologique** — aucune métrique n'y intervient —, il ne reste donc au corrélateur que la classe d'isotopie de la courbe : un invariant de nœud, le polynôme de Jones (Witten 1989). La loi d'aire du §5 et le polynôme du §7 sont les deux visages d'une seule observable, l'un métrique, l'autre topologique — aucun des deux ne se déduit de l'autre.

</details>

## Pour aller plus loin

- **Carnets précédents** : [Jauge-02 — Invariance de jauge U(1) et SU(2)](Jauge-02-Invariance-Jauge-U1-SU2-Python.ipynb) (la symétrie continue, vérifiée symboliquement et numériquement) et [Jauge-01 — Théorème de Noether](Jauge-01-Theoreme-Noether-Python.ipynb) (le socle : symétrie ⇒ courant conservé).
- **La voie sœur sur les mêmes polynômes** : [KNOTS-02 — Invariants de nœuds](../../SymbolicAI/Lean/Noeuds/KNOTS-02-Invariants-Python.ipynb) — le polynôme de Jones par le crochet de Kauffman et l'algèbre de Temperley–Lieb : la première des deux voies indépendantes que le §7 recoupe, avec la spécialisation aux niveaux $k$ et les mêmes trois nœuds.
- **Suite prévue de la série** : formulation — développement en caractères et dualité exacte des modèles de jauge abéliens en dimension 2 (gaz de Coulomb, gaz de vortex), annoncée en conclusion.
- **La lecture exhaustive de la soumission** : [cartographie sourcée de la soumission Yang–Mills OpenAI du 28/10/2026](../../../docs/research/yang-mills-openai-20261028-analyse.md) — ce qui est formalisé (et où, `fichier:ligne`), ce qui est hors périmètre, et l'outil rejouable [`scripts/ym_openai_20261028/inventory.py`](../../../scripts/ym_openai_20261028/inventory.py) qui régénère l'inventaire depuis un clone local.
- **Fil de la série** : [#23161 — EPIC Série Physique numérique](https://github.com/jsboige/CoursIA/issues/23161) — arc Noether → Yang–Mills.
- **Sources** : K. G. Wilson, *Confinement of quarks*, Physical Review D **10**, 2445 (1974) · M. Creutz, *Monte Carlo study of quantized SU(2) gauge theory*, Physical Review D **21**, 2308 (1980) · E. Witten, *Quantum Field Theory and the Jones Polynomial*, Communications in Mathematical Physics **121**, 351–399 (1989) — archivée au gisement : `G:\Mon Drive\MyIA\IA\Bibliographie IA\Physics\1989 - Witten - Quantum Field Theory and the Jones Polynomial (Comm. Math. Phys. 121, 351-399).pdf`. La construction R-matrice / trace quantique suit la présentation standard de $U_q(\mathfrak{sl}_2)$ ; les valeurs de référence sont celles du Knot Atlas (`3_1`, `4_1`, `5_1`).

***

**Jumeau C#** de [Jauge-03 — L'action de Wilson sur réseau : la boucle de Wilson et le confinement (Python)](Jauge-03-Action-Wilson-Confinement-Python.ipynb) · BCL seule, sans paquet externe · série Physique / Jauge.